# 03 · FunnyBird standard MCBM — does minimality repair concept grounding?

**Report question.** Notebook 02 discovered controlled concept backwash in a
standard CBM. When MCBM increasingly penalizes information in each internal
concept slot beyond its binary label, do the same validated part replacements
become more correctly attributed?

**Population.** Standard non-RLv2 MCBM at gamma `0, 0.1, 0.3, 1, 3, 5`, with
standard CBM retained only as the discovery reference. The all-gamma fixed-render
causal comparison currently has one independently trained seed per gamma.

**Claims available here.** This notebook can test whether gamma compresses the
implemented internal representation and whether that compression repairs the
already-defined FunnyBird controlled event. It cannot attribute standard-CBM
versus MCBM-gamma-zero differences to minimality, call one causal seed a stable
gamma curve, or replace the earlier standard-CBM discovery with an MCBM result.

This is the non-RLv2 MCBM stage. RLv2 is a later causal label test.

**Part names are outcomes, not mechanisms.** The general hypothesis is a
competition between the original-image source advantage and a response driven
by the changed part pixels. The starting advantage is not pure context because
the original source part is still present; later species-residual tests ask
whether context helps maintain it after replacement. The report
must evaluate all five parts without presupposing their ordering. Notebook 06 must establish its own CUB ordering
from all exact concepts and masks; it must not presume that CUB tail is special.

More precisely, the proposed contributors are properties of each
part/concept and its data: the original source-versus-donor margin, the size of
the response to the inserted pixels, positive-label/visibility conflict,
exact-value difficulty, the number and frequency of alternatives, and residual
source-species organization. Backwash should be strongest wherever these
properties combine unfavourably. Whether these contributors predict the observed
ordering is a question for the current outputs, not an assumed result.


## Standard CBM versus MCBM: different wiring, related questions

The two primary papers do **not** implement the same network with one extra
loss term.

**Accepted Koh Joint Standard CBM from notebook 02**

```text
image x -> ResNet-50 -> 26 raw concept logits z
                              |-> sigmoid(z_j) for concept j
                              `-> one linear layer Wz+b -> 50 species logits
```

The species layer reads the exact 26 raw concept logits studied in notebook 02.
The accepted training objective is

`L_Koh = [L_species(Wz+b,y) + 0.01 * sum_j L_concept,j(z_j,c_j)] / [1 + 0.01*26]`.

Here `L_species` is 50-class cross-entropy and each `L_concept,j` is Koh's
weighted binary-logit loss. During the accepted `use_aux` training, each species
and concept term also includes 0.4 times its auxiliary-output loss; the displayed
equation abbreviates that main-plus-auxiliary term, not an auxiliary-free recipe.
The final denominator is the behavior selected by
Koh's `normalize_loss` option. This is Koh's Joint CBM with the approved
ResNet-50 encoder substitution. It is not the model Koh separately called
`Standard`, which has no concept loss.

**Official minimal_cbm MCBM used here**

```text
image x -> ResNet-50 -> 26 internal scalars h
                              |-> one learned 1 -> 3 -> 1 concept head per j
                              |      q_j(h_j) = raw concept logit z_j
                              |      sigmoid(z_j) = concept probability
                              `-> learned 26 -> 256 -> 50 species MLP
```

During MCBM training only, the implementation feeds
`h_tilde = h + epsilon`, `epsilon ~ Normal(0,I)`, to both readers. Evaluation
uses `h` without sampled noise. The pinned implementation minimizes

`L_MCBM = L_species + beta * L_concept + gamma * L_rep`, with `beta=1`,

`L_rep = 0.2 * sum_j mean_i[(h_ij - (6*c_ij-3))^2]`.

Thus `c_ij=0` gives target `-3`, and `c_ij=1` gives target `+3`. This is the
repository's concrete mean-squared-error form of the MCBM paper's variational
regularizer. The paper derives a KL penalty and fixes binary prototypes at
`-lambda/+lambda` with `lambda=3`.

Primary sources: [Koh et al. (2020)](https://proceedings.mlr.press/v119/koh20a.html)
and [Almudévar et al. (2026)](https://arxiv.org/abs/2506.04877). The implemented
equations are additionally checked against the pinned `train.py`, `mcbm.py`,
`cbm.py`, and `vanilla.py` source files before this report is built.

| Symbol | Meaning |
|---|---|
| `x_i` | image `i` |
| `y_i` | species label for image `i` |
| `c_ij` | processed binary label for exact concept `j` |
| `h_ij` | MCBM encoder's internal scalar slot for concept `j`; the MCBM species MLP reads the complete vector `h_i` |
| `q_j` | learned `1 -> 3 -> 1` concept head for exact concept `j` |
| `z_ij=q_j(h_ij)` | post-head raw concept logit; the primary grounding score |
| `p_ij=sigmoid(z_ij)` | bounded probability, used only for thresholded performance |
| `c_hat_ij=1[z_ij>0]` | thresholded concept prediction |
| `gamma` | weight on MCBM's representation-compression loss |

The minimal-CBM source code calls the internal tensor `z`, but this report calls
it `h` because it is not yet the post-head concept logit. Every grounding figure
uses the post-head raw score `z_ij`. Ordinary accuracy and recall measure label
agreement; they do not reveal which image pixels produced the score.

Gamma pushes each MCBM internal slot toward its binary-label target. It does
**not** tell the encoder which pixels to use. A species/body shortcut can
predict the right label and be compressed neatly to `-3/+3`. Compression is
therefore evidence that the new loss acted, not evidence that grounding improved.

**Why add h and a separate reader q_j?** MCBM regularizes an internal code h_j,
then learns how that code predicts the named binary answer. Koh instead passes
the concept logit itself to its species reader. These are distinct designs,
not interchangeable variable names. The MCBM paper motivates minimizing
conditional information I(H_j;X|C_j): how much an internal slot can still tell
us about the image once the concept label is already known. For example, among
tail_4-positive images, h=3 for every image leaves no magnitude difference to
identify species; h=2 for one species and4 for another leaves such a clue.
Its Gaussian variational penalty leads to the implemented squared distance
from label-conditioned prototypes. This finite objective and a finite decoder
test do not certify that every possible species clue has disappeared.

The pinned implementation uses the fixed mapping c→6c−3 for its target, not a
learned image-dependent target. The learned q_j maps h_j to z_j. Its separate
species MLP reads all h slots jointly. The paper's internal concept corrections
and information diagnostics test useful representation properties, but do not
replace our physical-image swap test of where the encoder got its evidence.

### Important: neither `h` nor raw `z` is clipped to `[-3,+3]`

The values `-3` and `+3` are **targets in a squared-error penalty**, not hard
bounds. An internal slot `h_ij` may still be smaller than `-3` or larger than
`+3`, especially when gamma is small. The learned `1 -> 3 -> 1` concept head
then maps `h_ij` to the final raw logit `z_ij`; that output is also unbounded.
Only `p_ij=sigmoid(z_ij)` lies between zero and one.

The older exploratory FunnyBird MCBM notebooks used `z` for a different
intermediate quantity and sometimes compared a CBM sigmoid probability with an
MCBM raw score. Their useful questionâ€”separating the pre-swap preference from
the change caused by the swapâ€”is restored below using the current checkpoints
and four verified raw logits. Their raw numerical scales are not reused.

Because separately trained heads can use different raw-logit scales, raw
magnitudes are interpreted primarily **within a model**. Cross-model claims rely
most strongly on predicates, fractions, exact-value ranks, and the fraction of
the model's own starting deficit that the swap closes. A larger raw-logit change
in one model is not automatically a stronger cross-model effect.

There are several baseline differences: MCBM has nonlinear concept heads, a
nonlinear species head, Gaussian training noise, different concept-loss
weighting, and an independently optimized checkpoint. Consequently:

- standard CBM versus MCBM `gamma=0` tests the training-noise/optimization
  baseline because the minimality loss has zero weight;
- MCBM `gamma=0` versus positive gamma tests the added minimality pressure.

An observed Koh-versus-MCBM-gamma-zero difference cannot be credited to
minimality. Gamma-zero-to-positive-gamma changes within MCBM are the closer regularizer comparison,
but independent optimization and limited seed coverage remain competing explanations.

For a controlled replacement from source value `s` to donor value `d`:

- `m_orig=z_donor,orig-z_source,orig` is the donor-minus-source margin before replacement;
- `m_cf=z_donor,cf-z_source,cf` is the same margin after replacement;
- `response_delta=m_cf-m_orig` is movement caused by the changed image;
- controlled backwash is `response_delta>0 and m_cf<0`.

Example: `m_orig=-20` and `m_cf=-5` gives `response_delta=+15`. The donor pixels
moved the comparison 15 raw-logit units toward the donor, but the old source
still finishes 5 units higher. That is a controlled backwash event.


In [ ]:
# ALT: Imports and shared plotting definitions; no scientific figure.
import os, re, glob, json, sys, hashlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
CURATED=Path(os.environ["CURATED_DATA"])
REPO=Path.cwd() if (Path.cwd()/"analysis").is_dir() else Path.cwd().parent
KOH_MODEL_ROOT=CURATED/"koh_joint_resnet_accelerated_converged_v1"/"funnybirds"/"standard"/"seed1"
KOH_SWAP_ROOT=CURATED/"swap_koh_joint_resnet_accelerated_converged_v1_seed1"
MCBM_FIXED_ROOT=CURATED/"swap_fixed_v2_attempt2"
VISIBILITY_ROOT=CURATED/"funnybird_visibility_correction_v1"
sys.path.insert(0,str(REPO/"analysis"))
from minimal_cbm_scores import concept_logits_from_saved_latent, validate_saved_probabilities
ORDER=["tail","wing","beak","foot","eye"]
COLORS=dict(tail="#7B3294",wing="#0080C6",beak="#E66101",foot="#009E73",eye="#CC79A7")
GAMMAS=[0.,.1,.3,1.,3.,5.]
plt.rcParams.update({"figure.dpi":120,"axes.grid":False})

def heat(ax, table, title, cbar, vmin=None, vmax=None, cmap="viridis", fmt=".2f"):
    a=table.astype(float).values
    if vmin is None: vmin=np.nanmin(a)
    if vmax is None: vmax=np.nanmax(a)
    im=ax.imshow(a,aspect="auto",cmap=cmap,vmin=vmin,vmax=vmax)
    ax.set_xticks(range(len(table.columns))); ax.set_xticklabels(table.columns)
    ax.set_yticks(range(len(table.index))); ax.set_yticklabels(table.index)
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            if np.isfinite(a[i,j]): ax.text(j,i,format(a[i,j],fmt),ha="center",va="center",fontsize=8)
    ax.set_title(title); plt.colorbar(im,ax=ax,label=cbar,fraction=.046)


## Dataset, population, and causal capability

FunnyBird has 50 species, 26 exact concept values, and five named parts:
`tail`, `wing`, `beak`, `foot`, and `eye`. The accepted renderer changes one
part while holding body, pose, camera, and background fixed. That makes
`response_delta` and `m_cf` causal same-image measurements of the inserted
pixels. Visibility, value support, and source-species residuals remain proposed
contributors unless independently manipulated.

All MCBM gamma comparisons use epoch 100 and the accepted MCBM fixed-render
root `swap_fixed_v2_attempt2`. Whenever Koh Standard is shown, it comes from
the accepted converged Koh root
`swap_koh_joint_resnet_accelerated_converged_v1_seed1`, never from the
`minimal_cbm` repository's different CBM class. Figure 1 verifies that the two
roots identify the same replacement pixels before combining them.

| Item | Value used here | Why it matters |
|---|---:|---|
| species | 50 | unchanged source-species/body appearance is a possible contextual signal |
| named parts | `tail`, `wing`, `beak`, `foot`, `eye` | exactly the same five interventions as notebook 02 |
| exact concepts | 26 part values | exact-value confusion can be separated from coarse part identity |
| ordinary held-out population | 5,000 images per available seed | used for health, compression, decoding, and recall diagnostics |
| fixed-render population | 5,000 directed replacements per gamma at seed 1 | used for causal gamma comparisons |

Notebook 02 already displayed and accepted the semantic renderer preflight for
all five parts. Figure 1 below does not replace that visual inspection: it proves
that every gamma uses those same accepted counterfactual render IDs and byte
hashes. The invalid black-render cache and uncalibrated deletion/patch methods
are not loaded anywhere in this report.


## 1 · What data, checkpoints, renders, gammas, and seeds are actually compared?

**Notebook 02 connection.** Notebook 02 first established model health and the
renderer intervention. MCBM adds a gamma sweep, so this report must additionally
prove that every gamma sees the same counterfactual pixels.

**Question.** Is every gamma evaluated on the same validated pixels, and how many
independent seeds support each result?

**Variables and prediction.** A valid row must contain 5,000 unique directed
replacement IDs, all five parts, both directions, all 50 source and donor
species, finite raw logits, and the same render-ID-to-byte-hash mapping as every
other gamma. The stored `margin`, `margin_orig`, and `response_delta` must agree
with values recomputed from the four raw logits.

**Method and exclusions.** The runner first executes the repository's complete
fixed-render validator. This cell then independently checks schema, finiteness,
algebra, identities, checkpoint existence, and file hashes. A non-finite or
unfinished checkpoint is not silently counted as a seed.

### Figure 1 · Are the gamma comparisons mechanically matched?

**How to read the figure.** Each row is one gamma/seed CSV. `rows` is the number
of directed swaps; `render_ids` is the number of unique counterfactual images;
`parts`, `directions`, and the species columns describe coverage. `csv_sha256` and
`checkpoint_sha256` identify the exact inputs. `render_ids` is the number of
unique counterfactual images. `max_algebra_error` is the largest disagreement
between saved and recomputed margins; values near zero are expected. Example:
5,000 rows, five parts, and two directions means 500 swaps per part and direction.
This is an input audit, not a model result.


In [ ]:
# ALT: Figure 1. Inventory of validated fixed-render MCBM comparisons by gamma and seed.
FIXED=MCBM_FIXED_ROOT
if not FIXED.exists(): raise FileNotFoundError(f"validated fixed-render directory missing: {FIXED}")

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()

KOH_CSV=KOH_SWAP_ROOT/"funnybirds-cbm-s1.csv"
if not KOH_CSV.exists():
    raise FileNotFoundError(f"accepted converged Koh swap CSV missing: {KOH_CSV}")
CB=pd.read_csv(KOH_CSV).assign(seed=1,source_csv=KOH_CSV.name)
required_identity={"render_id","image_cf_sha256","image_orig_sha256","part"}
if required_identity-set(CB.columns):
    raise RuntimeError(f"Koh CSV lacks identity columns {sorted(required_identity-set(CB.columns))}")
if CB.render_id.duplicated().any():
    raise RuntimeError("accepted Koh CSV has duplicate render IDs")
reference_render_map={str(r.render_id):(str(r.image_cf_sha256),str(r.image_orig_sha256),str(r.part))
                      for r in CB.itertuples()}
reference_name=KOH_CSV.name

rows=[]; file_meta=[]
for fp in sorted(FIXED.glob("funnybirds-mcbm-g*-s*.csv")):
    m=re.fullmatch(r"funnybirds-mcbm-g([0-9p]+)-s(\d+)\.csv",fp.name)
    if not m: continue
    d=pd.read_csv(fp); g=float(m.group(1).replace("p",".")); seed=int(m.group(2))
    required={"part","direction","z_new","z_old","z_new_orig","z_old_orig",
              "margin","margin_orig","response_delta","sid_src","sid_donor",
              "render_id","image_cf_sha256","image_orig_sha256","var_src","var_donor"}
    missing=required-set(d.columns)
    if missing: raise RuntimeError(f"{fp.name} schema missing {sorted(missing)}")
    numeric=["z_new","z_old","z_new_orig","z_old_orig","margin","margin_orig","response_delta"]
    if not np.isfinite(d[numeric].to_numpy(float)).all():
        raise RuntimeError(f"{fp.name} contains non-finite grounding values")
    m_orig=d.z_new_orig-d.z_old_orig; m_cf=d.z_new-d.z_old; delta=m_cf-m_orig
    algebra=max(float(np.max(np.abs(d.margin-m_cf))),
                float(np.max(np.abs(d.margin_orig-m_orig))),
                float(np.max(np.abs(d.response_delta-delta))))
    if algebra>1e-6: raise RuntimeError(f"{fp.name} stored/recomputed margin mismatch: {algebra}")
    if set(d.part)!=set(ORDER) or set(d.direction)!={"fwd","bwd"}:
        raise RuntimeError(f"{fp.name} has wrong part or direction population")
    if d.render_id.duplicated().any(): raise RuntimeError(f"{fp.name} has duplicate render IDs")
    render_map={str(r.render_id):(str(r.image_cf_sha256),str(r.image_orig_sha256),str(r.part))
                for r in d.itertuples()}
    if render_map!=reference_render_map:
        raise RuntimeError(f"{fp.name} render IDs/bytes/parts differ from accepted {reference_name}")
    tag=m.group(1); ck=REPO/"external/minimal_cbm/results"/f"funnybirds-mcbm-g{tag}"/str(seed)/"models/epoch_100.pt"
    if not ck.exists(): raise FileNotFoundError(f"matching checkpoint missing: {ck}")
    d["gamma"]=g; d["seed"]=seed; d["source_csv"]=fp.name; rows.append(d)
    file_meta.append(dict(gamma=g,seed=seed,csv=fp.name,rows=len(d),
        render_ids=d.render_id.nunique(),parts=d.part.nunique(),directions=d.direction.nunique(),
        source_species=d.sid_src.nunique(),donor_species=d.sid_donor.nunique(),
        csv_sha256=sha256(fp)[:16],checkpoint=str(ck),checkpoint_sha256=sha256(ck)[:16],
        max_algebra_error=algebra))
if not rows: raise FileNotFoundError("no validated standard-MCBM fixed-render CSVs")
SW=pd.concat(rows,ignore_index=True)
SW["m_orig"]=SW.z_new_orig-SW.z_old_orig
SW["m_cf"]=SW.z_new-SW.z_old
SW["response_delta"]=SW.m_cf-SW.m_orig
SW["backwash"]=(SW.response_delta>0)&(SW.m_cf<0)
CB["m_orig"]=CB.z_new_orig-CB.z_old_orig
CB["m_cf"]=CB.z_new-CB.z_old
CB["response_delta"]=CB.m_cf-CB.m_orig
CB["backwash"]=(CB.response_delta>0)&(CB.m_cf<0)

VISIBILITY_TABLE=VISIBILITY_ROOT/"visibility.csv"
if not VISIBILITY_TABLE.exists():
    raise FileNotFoundError(f"corrected visibility table missing: {VISIBILITY_TABLE}")
visibility=pd.read_csv(VISIBILITY_TABLE)
if visibility.duplicated(["render_id","part"]).any():
    raise RuntimeError("corrected visibility key is not unique")
for name,frame in [("Koh",CB),("MCBM",SW)]:
    before=len(frame)
    merged=frame.merge(
        visibility[["render_id","part","legacy_single_instance_pixels",
                    "corrected_all_instance_pixels","added_second_instance_pixels"]],
        on=["render_id","part"],how="left",validate="many_to_one")
    if len(merged)!=before or merged.corrected_all_instance_pixels.isna().any():
        raise RuntimeError(f"corrected visibility does not cover every {name} swap row")
    merged["pixel_count_cf_legacy_single_instance"]=merged["pixel_count_cf"]
    merged["pixel_count_cf"]=merged.corrected_all_instance_pixels.astype(int)
    if name=="Koh": CB=merged
    else: SW=merged
inv=pd.DataFrame(file_meta).sort_values(["gamma","seed"])
if set(inv.gamma)!=set(GAMMAS): raise RuntimeError(f"expected gamma set {GAMMAS}; got {sorted(inv.gamma.unique())}")
display(inv)
print("fixed render root:",FIXED)
print("accepted Koh comparison:",KOH_CSV)
print("corrected visibility:",VISIBILITY_TABLE)


### Review record for Figure 1

**INCOMPLETE: current output requires execution and visual review.**

- **Literal result:** Record the actual values, sample sizes and exceptions.
- **What it supports:** State only the claim measured by this figure.
- **Plausible alternative:** Give a concrete competing explanation.
- **Discriminating test:** State which observation would distinguish it.
- **Next question:** Explain why the next analysis follows from this result.

Display the complete current figure in chat before filling this record.


## 2 · Did gamma compress the intended internal slots without breaking prediction?

**Notebook 02 connection.** Notebook 02 checked whether standard-CBM concept
outputs were usable. This section repeats that guard and adds the MCBM-specific
question: did gamma actually enforce the representation penalty?

**Question.** Does increasing gamma move internal slots toward their label
targets and remove within-label variation without breaking ordinary prediction?

**Variables and prediction.** `target RMSE` is the root mean squared distance from each saved
internal slot `h_ij` to its `+3/-3` label target. `within-label spread` is the
median across concepts and labels of `Q95(h)-Q05(h)`. Lower means stronger
compression. Species accuracy and concept balanced accuracy are health checks.
Gamma should lower the first two quantities. A grounding claim is
interpretable only if species/concept health remains usable. These panels may use
all available checkpoints; dots are independent seeds and lines connect only
gamma means.

**Method and exclusions.** Replay every finite epoch-100 prediction/checkpoint
pair. Recompute post-head logits from saved `h`, verify the replayed probabilities,
and exclude unfinished or non-finite artifacts rather than counting them as seeds.

### Figure 2 · Compression and ordinary prediction health across gamma

**How to read the figure.** The x-axis in every panel is gamma. Orange dots are
independently trained seeds; the black point and line are the mean at each gamma.
Panel A is target RMSE in `h` units and Panel B is within-label `h` spread; lower
means stronger compression. Panel C is species accuracy and Panel D is concept
balanced accuracy; higher means healthier prediction. A fall from RMSE 20 to 1
means the slot is much closer to ±3. It does not mean the slot used the right
pixels.


In [ ]:
# ALT: Figure 2. Compression and ordinary prediction health across gamma; dots are independently trained seeds.
import torch
sys.path.insert(0,str(REPO/"data/funnybirds"))
import funnybirds_concepts as fbc
FB_ROOT=Path(os.environ.get("FUNNYBIRDS_ROOT",CURATED/"FunnyBirds"))
FB_PARTS=fbc.load_parts(FB_ROOT); CONCEPT_NAMES=fbc.concept_names(FB_PARTS); SPANS=fbc.group_slices(FB_PARTS)
CONCEPT_PART={name:part for part,(lo,hi) in SPANS.items() for name in CONCEPT_NAMES[lo:hi]}
health=[]; health_exact=[]; excluded_health=[]; HEALTH_DATA={}
for g,tag in [(0,"g0"),(.1,"g0p1"),(.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
  base=REPO/"external/minimal_cbm/results"/f"funnybirds-mcbm-{tag}"
  for sd in sorted(base.glob("[0-9]*")) if base.exists() else []:
    pp=sd/"predictions/epoch_100.pth"
    ck=sd/"models/epoch_100.pt"
    if not (pp.exists() and ck.exists()): continue
    d=torch.load(pp,map_location="cpu",weights_only=False); h=d["z"].float().reshape(len(d["z"]),-1); c=d["c"].float().reshape(len(h),-1)
    if not (torch.isfinite(h).all() and torch.isfinite(c).all()):
      excluded_health.append(dict(gamma=g,seed=int(sd.name),status="INVALID OUTPUT",reason="non-finite saved internal slots or labels")); continue
    logits=concept_logits_from_saved_latent(h,ck,c.shape[1])
    if not torch.isfinite(logits).all():
      excluded_health.append(dict(gamma=g,seed=int(sd.name),status="INVALID OUTPUT",reason="non-finite replayed concept logits")); continue
    err=validate_saved_probabilities(logits,d["c_preds"])
    target=6*c-3; rmse=float(((h-target)**2).mean().sqrt())
    spreads=[]
    for j in range(c.shape[1]):
      for lab in [0,1]:
        q=h[c[:,j]==lab,j]
        if len(q)>5: spreads.append(float(torch.quantile(q,.95)-torch.quantile(q,.05)))
    pred=(logits>0); tpr=((pred)&(c==1)).sum(0)/(c==1).sum(0).clamp(min=1); tnr=((~pred)&(c==0)).sum(0)/(c==0).sum(0).clamp(min=1)
    yp=d["y_preds"].reshape(len(h),-1); ya=float((yp.argmax(-1)==d["y"].reshape(-1)).float().mean())
    health.append(dict(gamma=g,seed=int(sd.name),target_rmse=rmse,within_label_spread=np.median(spreads),species_accuracy=ya,concept_balanced_accuracy=float(((tpr+tnr)/2).mean()),replay_error=err))
    HEALTH_DATA[(g,int(sd.name))]=dict(
      h=h.numpy(),c=c.numpy(),z=logits.numpy(),
      y=np.asarray(d["y"]).reshape(-1).astype(int),
      y_probability=np.asarray(d["y_preds"]).reshape(len(h),-1))
    for j,name in enumerate(CONCEPT_NAMES):
      zj=logits[:,j].numpy(); cj=c[:,j].numpy().astype(int); pj=zj>0
      pos=zj[cj==1]; neg=zj[cj==0]
      health_exact.append(dict(gamma=g,seed=int(sd.name),concept=name,part=CONCEPT_PART[name],
        spread=np.quantile(zj,.95)-np.quantile(zj,.05),
        full_range=np.max(zj)-np.min(zj),
        distinct_finite_scores=np.unique(zj[np.isfinite(zj)]).size,
        label_separation=np.median(pos)-np.median(neg),
        balanced_accuracy=.5*((pj[cj==1]).mean()+(~pj[cj==0]).mean()),
        positive_recall=(pj[cj==1]).mean()))
H=pd.DataFrame(health)
HEXACT=pd.DataFrame(health_exact)
if H.empty: raise FileNotFoundError("no standard MCBM prediction/checkpoint pairs")
display(H.round(4))
if excluded_health:
 print("Excluded unfinished/corrupt artifacts; these are not seeds:")
 display(pd.DataFrame(excluded_health))
fig,ax=plt.subplots(1,4,figsize=(15,3.5)); metrics=[("target_rmse","target RMSE h vs ±3"),("within_label_spread","within-label h spread"),("species_accuracy","species accuracy"),("concept_balanced_accuracy","concept balanced accuracy")]
for a,(metric,title) in zip(ax,metrics):
  for _,r in H.iterrows(): a.scatter(r.gamma,r[metric],color="#D55E00",alpha=.55)
  q=H.groupby("gamma")[metric].mean(); a.plot(q.index,q.values,"o-",color="black"); a.set_xlabel("gamma"); a.set_title(title)
plt.tight_layout()


### Review record for Figure 2

**INCOMPLETE: current output requires execution and visual review.**

- **Literal result:** Record the actual values, sample sizes and exceptions.
- **What it supports:** State only the claim measured by this figure.
- **Plausible alternative:** Give a concrete competing explanation.
- **Discriminating test:** State which observation would distinguish it.
- **Next question:** Explain why the next analysis follows from this result.

Display the complete current figure in chat before filling this record.


## 2b · Did any exact concept become unusable while the average stayed high?

**Notebook 02 connection.** This is the all-exact-concept health guard from
notebook 02, repeated separately at every MCBM gamma.

**Question.** Figure 2 averages across 26 concepts. Does that hide a constant or
broken exact output?

**Variables and prediction.** For exact concept `j`, `spread_j=Q95(z)-Q05(z)`,
`label_separation_j=median(z|c=1)-median(z|c=0)`, balanced accuracy gives positive
and negative labels equal weight, and `positive_recall_j=P(z>0|c=1)`. Here
`spread_j` describes the middle 90% of scores. It is not enough to prove that
every score is constant. We therefore also print `full_range=max(z)-min(z)` and
the number of distinct finite scores whenever `spread_j <= 1e-8`. Exact collapse
requires `full_range <= 1e-8`. Higher spread is not inherently better; it only
shows that scores vary.

Balanced accuracy is used because most exact concepts are absent from most
images. It is `(positive recall + negative recall)/2`. Example: if a concept is
positive in only 5% of images, predicting “absent” for every image gives 95%
ordinary accuracy but 50% balanced accuracy: it found none of the positives.

**Method and exclusions.** Use seed 1 for the gamma-aligned panels and print all
26 concepts. Non-finite checkpoints were already excluded in Figure 2.

### Figure 2b · Exact-concept health at every gamma

**How to read the figure.** Rows are exact concepts in the same order in all four panels;
columns are all six gammas. Positive label separation, balanced accuracy above
0.5, and positive recall above 0.5 are the expected health directions. These are
health checks, not evidence that the named pixels produced `z`. Example: balanced
accuracy 0.50 means the exact output gives no better-than-chance balanced binary
decision even if another panel shows high overall average accuracy.


In [ ]:
# ALT: Figure 2b. Exact-concept raw-z spread, label separation, balanced accuracy, and positive recall for all six MCBM gammas.
E=HEXACT[HEXACT.seed==1].copy()
metrics=[("spread","raw-z spread"),("label_separation","positive - negative median z"),
         ("balanced_accuracy","balanced accuracy"),("positive_recall","positive recall")]
fig,axes=plt.subplots(1,4,figsize=(18,max(7,.25*len(CONCEPT_NAMES))),sharey=True)
for ax,(metric,title) in zip(axes,metrics):
  T=E.pivot(index="concept",columns="gamma",values=metric).reindex(index=CONCEPT_NAMES,columns=GAMMAS)
  heat(ax,T,title,metric,0 if metric in ["spread","balanced_accuracy","positive_recall"] else None,
       1 if metric in ["balanced_accuracy","positive_recall"] else None,
       "viridis" if metric=="spread" else "coolwarm")
  ax.set_yticklabels(CONCEPT_NAMES,fontsize=7)
plt.tight_layout(); display(E.round(3))
central_zero=E[E.spread.le(1e-8)][
  ["gamma","concept","spread","full_range","distinct_finite_scores",
   "balanced_accuracy","positive_recall"]]
print("gamma/concept cells with zero central-90% spread:")
display(central_zero)
exact_collapsed=E.full_range.le(1e-8)
print("exact full-range-collapsed gamma/concept outputs:",int(exact_collapsed.sum()))


### Review record for Figure 2b

**INCOMPLETE: current output requires execution and visual review.**

- **Literal result:** Record the actual values, sample sizes and exceptions.
- **What it supports:** State only the claim measured by this figure.
- **Plausible alternative:** Give a concrete competing explanation.
- **Discriminating test:** State which observation would distinguish it.
- **Next question:** Explain why the next analysis follows from this result.

Display the complete current figure in chat before filling this record.


## 2c · Where does MCBM compression occur, and how does the learned concept head transform it?

**Notebook 02 connection.** Standard CBM has no ±3 representation target, so
this is an MCBM-specific mechanism test rather than a duplicated grounding plot.

**Question.** Does gamma compress every part similarly, and does the learned
concept head `q_j` compensate by amplifying small changes in the compressed
internal slot?

**Variables and prediction.** For each part and gamma, compute: (A) target RMSE
`sqrt(E[(h-(6c-3))^2])`; (B) within-label spread `median(Q95(h)-Q05(h))`; (C)
the mean absolute local head slope `E[|dz/dh|]`; and (D) the fraction of held-out
rows on a locally flat head branch, `P(|dz/dh|<=10^-4)`. The slope is estimated
by a centered finite difference of the saved learned head. If gamma merely
shrinks `h` but the head compensates, Panels A-B should fall while Panel C
stays large or rises. If tail loses head sensitivity, its Panel-C value should
fall and/or its Panel-D flat fraction should rise relative to other parts.

**Method and exclusions.** Use the same held-out seed-1 predictions and finite
checkpoints accepted in Figures 2-2b. Perturb every scalar `h_ij` by `±0.001`
and replay the exact saved concept heads. This measures local head behavior on
ordinary held-out images; it does not recover the unrecorded counterfactual
change in `h` and therefore cannot replace `response_delta`.

### Figure 2c · Per-part compression and concept-head sensitivity

**How to read the figure.** Rows are gamma and columns are the five parts in the
same order used throughout notebooks 02 and 03. Lower values in Panels A-B mean
stronger compression. In Panel C, mean `|dz/dh|=2` means that a local change of
`0.5` in `h` changes `z` by about `1` on average. Panel D is the fraction of
held-out image-concept rows for which the learned head is locally flat; larger
is less locally responsive. The printed table also separates positive,
negative, and flat slopes. This matters because a median slope of zero can hide
a responsive minority on a piecewise-linear ReLU head. These panels explain
where a score scale can change; they do not show which image pixels changed `h`.


In [ ]:
# ALT: Figure 2c. Per-part MCBM target compression, within-label internal variation, mean learned-head sensitivity, and locally flat-row fraction.
eps=1e-3
mechanism=[]
for g,tag in [(0,"g0"),(.1,"g0p1"),(.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    d=HEALTH_DATA[(g,1)]
    h=torch.as_tensor(d["h"],dtype=torch.float32)
    c=torch.as_tensor(d["c"],dtype=torch.float32)
    ck=REPO/"external/minimal_cbm/results"/f"funnybirds-mcbm-{tag}"/"1"/"models/epoch_100.pt"
    slope=((concept_logits_from_saved_latent(h+eps,ck,c.shape[1])-
            concept_logits_from_saved_latent(h-eps,ck,c.shape[1]))/(2*eps)).numpy()
    hn=h.numpy(); cn=c.numpy(); target=6*cn-3
    for part in ORDER:
        lo,hi=SPANS[part]; part_spreads=[]
        for j in range(lo,hi):
            for lab in [0,1]:
                q=hn[cn[:,j]==lab,j]
                if len(q)>5: part_spreads.append(np.quantile(q,.95)-np.quantile(q,.05))
        local=slope[:,lo:hi].reshape(-1)
        slope_tol=1e-4
        active=np.abs(local)>slope_tol
        mechanism.append(dict(
            gamma=g,part=part,
            target_rmse=np.sqrt(np.mean((hn[:,lo:hi]-target[:,lo:hi])**2)),
            within_label_h_spread=np.median(part_spreads),
            mean_abs_dz_dh=np.mean(np.abs(local)),
            active_median_abs_dz_dh=(np.median(np.abs(local[active])) if active.any() else 0.0),
            positive_slope_fraction=np.mean(local>slope_tol),
            negative_slope_fraction=np.mean(local < -slope_tol),
            flat_slope_fraction=np.mean(~active)))
MECHANISM=pd.DataFrame(mechanism)
fig,ax=plt.subplots(1,4,figsize=(18,4))
spec=[("target_rmse","A. Distance from ±3 target","h RMSE",0,None,"viridis"),
      ("within_label_h_spread","B. Remaining within-label h variation","Q95-Q05 in h",0,None,"viridis"),
      ("mean_abs_dz_dh","C. Mean learned-head local sensitivity","mean |dz/dh|",0,None,"viridis"),
      ("flat_slope_fraction","D. Locally flat learned-head rows","fraction |dz/dh| <= 1e-4",0,1,"magma_r")]
for a,(metric,title,label,vmin,vmax,cmap) in zip(ax,spec):
    T=MECHANISM.pivot(index="gamma",columns="part",values=metric).reindex(index=GAMMAS,columns=ORDER)
    heat(a,T,title,label,vmin,vmax,cmap)
plt.tight_layout(); display(MECHANISM.round(4))


### Review record for Figure 2c

**INCOMPLETE: current output requires execution and visual review.**

- **Literal result:** Record the actual values, sample sizes and exceptions.
- **What it supports:** State only the claim measured by this figure.
- **Plausible alternative:** Give a concrete competing explanation.
- **Discriminating test:** State which observation would distinguish it.
- **Next question:** Explain why the next analysis follows from this result.

Display the complete current figure in chat before filling this record.


## 2d · Does the historically flagged tail output create the tail gamma result?

**Notebook 02 connection.** Notebook 02 required exact-output health before
interpreting a part-level swap average. The previous report flagged one exactly constant
output (verify against the current Figure 2b): `tail_7` at gamma zero. This sensitivity analysis prevents that one
broken output from silently determining the MCBM conclusion.

**Question.** Does the tail gamma pattern remain after removing every controlled
swap whose source or donor tail value is 7?

**Variables and prediction.** For the complete tail population and the matched
population excluding value 7, report mean `response_delta`, median final margin
`m_cf`, controlled-backwash rate `P(response_delta>0 and m_cf<0)`, and exact
donor-value recognition. If the collapsed output created the result, removing
value 7 should strongly reduce or reverse the gamma trend. If both lines retain
the same ordering, the tail result is broader than that output.

**Method and exclusions.** Apply the same exclusion to every gamma, even though
only gamma zero has the exact collapse, so every line uses the same set of tail
value pairs. No images, thresholds, or model outputs are changed.

### Figure 2d · Tail gamma results with and without value 7

**How to read the figure.** The blue line includes all tail swaps; the orange
line excludes swaps with source value 7 or donor value 7. Each point is the
seed-1 mean or median over the printed number of fixed-render rows. In Panels A
and B, larger is better. In Panel C, lower controlled backwash is better. In
Panel D, larger exact donor recognition is better. Example: if both Panel-C
lines rise after gamma zero, value 7 cannot be the sole cause of worsening.


In [ ]:
# ALT: Figure 2d. Tail response, final margin, controlled-backwash rate, and exact donor-value recognition before and after excluding every value-7 swap.
tail=SW[SW.part.eq("tail") & SW.seed.eq(1)].copy()
tail_rows=[]
for g in GAMMAS:
    d=tail[tail.gamma.eq(g)]
    for population,q in [
        ("all tail swaps",d),
        ("exclude source/donor value 7",d[(d.var_src.ne(7)) & (d.var_donor.ne(7))])]:
        cols=sorted([c for c in q if c.startswith("z_cf_tail_")],
                    key=lambda x:int(x.rsplit("_",1)[1]))
        donor=q.var_donor.astype(int).to_numpy()
        pred=q[cols].to_numpy().argmax(1)
        valid=(donor>=0)&(donor<len(cols))
        tail_rows.append(dict(
            gamma=g,population=population,n=len(q),
            mean_response_delta=q.response_delta.mean(),
            median_final_margin=q.m_cf.median(),
            controlled_backwash_rate=q.backwash.mean(),
            exact_donor_recognition=float((pred[valid]==donor[valid]).mean())))
TAIL7_SENSITIVITY=pd.DataFrame(tail_rows)
fig,ax=plt.subplots(1,4,figsize=(16,3.6))
metrics=[
  ("mean_response_delta","A. Donorward movement","mean response_delta"),
  ("median_final_margin","B. Final donor-minus-source margin","median m_cf"),
  ("controlled_backwash_rate","C. Controlled backwash","fraction of swaps"),
  ("exact_donor_recognition","D. Exact donor-value recognition","fraction correct")]
for a,(metric,title,ylabel) in zip(ax,metrics):
    for population,color in [("all tail swaps","#4c78a8"),
                             ("exclude source/donor value 7","#f58518")]:
        q=TAIL7_SENSITIVITY[TAIL7_SENSITIVITY.population.eq(population)]
        a.plot(q.gamma,q[metric],marker="o",label=population,color=color)
    a.set(title=title,xlabel="gamma",ylabel=ylabel)
    a.set_xticks(GAMMAS)
    if metric in {"controlled_backwash_rate","exact_donor_recognition"}:
        a.set_ylim(0,1)
    a.axhline(0,color="black",lw=.7,alpha=.5)
ax[0].legend(frameon=False,fontsize=8)
plt.tight_layout(); display(TAIL7_SENSITIVITY.round(4))


### Review record for Figure 2d

**INCOMPLETE: current output requires execution and visual review.**

- **Literal result:** Record the actual values, sample sizes and exceptions.
- **What it supports:** State only the claim measured by this figure.
- **Plausible alternative:** Give a concrete competing explanation.
- **Discriminating test:** State which observation would distinguish it.
- **Next question:** Explain why the next analysis follows from this result.

Display the complete current figure in chat before filling this record.


## The loss-engineering story: same questions, new model

1. Did the regularizer compress **h**, and did q_j turn that compression into usable **z**?
2. On the **same changed pixels**, did the named concept become more correct? Keep
   donor wins, positive-but-insufficient response, no response, and ties separate.
3. Which alternatives survive: visibility, conflicting labels, exact-value recognition,
   support, or source-species organization? Test all parts, not only the expected tail.
4. What information remains available, what does the original species reader use,
   and which loss incentive could address the measured failure?

Below, each Standard construction is displayed first, followed by the same
construction separately for each gamma. Shared renderer/data figures are shown
once because the pixels and labels do not change with gamma. This is explicit
parity, not a claim that different architectures have identical raw-score units.

**Population warning:** Standard ordinary diagnostics use its 500 saved images;
MCBM ordinary exports use their own saved population (printed before each graph,
usually 5,000). They are not a matched-image Koh-versus-MCBM accuracy experiment.
Within MCBM, labels/order and split rules are checked across gamma. Swaps are the
same 5,000 directed rows from 250 originals. Fivefold diagnostics hold out ordinary
images; they do not hold out species or provide independent trained-model seeds.

**Scale warning:** raw h/z and class logits are model-specific units. Use within-model
changes and dimensionless outcome fractions for cross-model claims. Three-slot
sampling controls coordinate count, not decoder capacity or all information.

**Checklist scope:** every Standard figure/example/table is enumerated. Its three
result-writing cells are replaced by current MCBM tables and review rules, not copied
as observations. Its Koh setup and provenance cell are replaced by the MCBM input
audit and final execution ledger. Shared label-conflict counts describe the declared
Standard training-view population; they are candidate descriptors, not proof of a
loss mechanism or training-time identity of a historical MCBM artifact.


In [ ]:
# ALT: Input adapters and explicit Standard baseline display; no scientific figure.
from IPython.display import Markdown, Image as DisplayImage
from mcbm_loss_report import (task_head, checkpoint_tag, replacement_use, score_reference,
                             replay_counterfactual_h, off_target_erasure)
from funnybird_followup_diagnostics import (conditional_information, load_label_conflict,
    conflict_response_table, ordinary_value_recognition, add_descriptors, prediction_audit,
    value_holdout_audit)
# The standalone helper selects Agg for CLI figures; restore notebook display.
get_ipython().run_line_magic('matplotlib','inline')
parts=FB_PARTS
STANDARD_NOTEBOOK=REPO/'notebooks/02_funnybirds_cbm.ipynb'
STANDARD_RENDER=json.loads(STANDARD_NOTEBOOK.read_text(encoding='utf-8'))
PARITY_CACHE={g:{} for g in GAMMAS}
PARITY_RUN=[]
CONFLICT=load_label_conflict(CURATED,CONCEPT_NAMES,SPANS)
PART_CONFLICT=CONFLICT.groupby('part').agg(n_positive=('positive_images','sum'),
    n_changed=('hidden_positive_images','sum')).reindex(ORDER)
PART_CONFLICT['conflict_rate']=PART_CONFLICT.n_changed/PART_CONFLICT.n_positive
for g in GAMMAS:
    if (g,1) not in HEALTH_DATA:
        raise RuntimeError(f'Required gamma={g} seed-1 ordinary predictions unavailable')
    if HEALTH_DATA[(g,1)]['z'].shape[1]!=26:
        raise RuntimeError('Expected 26 post-head logits')
    if not np.array_equal(HEALTH_DATA[(g,1)]['c'],HEALTH_DATA[(0.,1)]['c']):
        raise RuntimeError('MCBM ordinary label populations/order differ across gamma')
    if not np.array_equal(HEALTH_DATA[(g,1)]['y'],HEALTH_DATA[(0.,1)]['y']):
        raise RuntimeError('MCBM ordinary species populations/order differ across gamma')

def show_standard(tag, expected_source_hash):
    candidates=[c for c in STANDARD_RENDER['cells'] if c['cell_type']=='code'
                and re.fullmatch('fb-'+re.escape(tag)+'-[0-9a-f]+',c.get('id',''))]
    if len(candidates)!=1: raise RuntimeError(f'Standard baseline figure missing: {tag}')
    cell=candidates[0]
    if hashlib.sha256(''.join(cell['source']).encode()).hexdigest()!=expected_source_hash:
        raise RuntimeError(f'Standard {tag} source changed: render current Notebook02 before Notebook03')
    if not cell.get('outputs'):
        raise RuntimeError(f'Standard {tag} has no executed output; run Notebook02 first')
    if any(o['output_type']=='error' for o in cell['outputs']):
        raise RuntimeError(f'Standard {tag} contains an execution error')
    print('STANDARD KOH BASELINE — saved current Notebook02 output:',tag)
    for output in cell['outputs']:
        if 'data' in output:
            display({key:''.join(value) if isinstance(value,list) else value
                     for key,value in output['data'].items()},raw=True)
        elif output.get('output_type')=='stream':
            print(''.join(output.get('text',[])),end='')
    PARITY_RUN.append(dict(figure=tag,model='Koh Standard',status='executed baseline displayed'))


## Standard construction f1

## 1 · Did training produce a usable, non-collapsed CBM?

**Question.** Did training produce a usable, non-collapsed CBM?

**Variables and prediction.** For every exact concept `j`, measure raw-score spread, positive-versus-negative label separation, balanced accuracy, and positive recall. A usable slot has nonzero spread, positive label separation, and above-chance thresholded performance.

**Method.** Compute all quantities from the accepted converged checkpoint's held-out predictions. Recall is a health statistic, not grounding evidence.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 1 · Did training produce a usable, non-collapsed CBM?

**How to read the figure.** Each row is one exact concept, such as `yellow tail`. The four panels use the
same rows. `spread = Q95(z)-Q05(z)` asks whether the output changes across
test images; exactly zero means a constant output. `label separation =
median(z|c=1)-median(z|c=0)` asks how far positive-labelled images sit above
negative-labelled images; positive is the expected direction. `balanced
accuracy = (positive recall + negative recall)/2` gives positive and negative
labels equal weight; 0.5 is chance for a binary concept. `positive recall =
P(z>0|c=1)` is the fraction of labelled-positive images called positive.
Example: positive recall 0.90 means 90 of 100 positive-labelled images have
`z>0`. Dot color identifies the FunnyBird part: purple tail, blue wing,
orange beak, green foot, and pink eye. The solid zero line marks no label
separation; the dashed 0.5 lines mark chance balanced accuracy and 50%
positive recall. These are health checks, not evidence about which pixels
produced `z`.


- **Method in one line:** We computed four health statistics directly from the frozen CBM's 26 raw concept logits on 500 held-out images; no classifier or model was fitted.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
show_standard('f1','f1b7ef30f54f623fe9ec34824a4aba08dca719b72d610e06167fa001022cc6c2')


In [ ]:
# ALT: MCBM gamma=0: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()

PARITY_RUN.append(dict(figure='f1',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()

PARITY_RUN.append(dict(figure='f1',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()

PARITY_RUN.append(dict(figure='f1',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()

PARITY_RUN.append(dict(figure='f1',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()

PARITY_RUN.append(dict(figure='f1',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()

PARITY_RUN.append(dict(figure='f1',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f2a

## 2 · Did the renderer change only the intended part?

**Question.** Did the renderer change only the intended part?

**Variables and prediction.** Inspect the semantic preflight and original/swap/delete/part-map examples for all five parts. For a visible replacement, the target part should change while the rest of the scene is preserved. Rows whose rendered RGB image does not change must remain identifiable and be handled by the later visibility analysis, not counted as visibly changed.

**Method.** Use the accepted full-cache validation plus representative all-part examples before reading any model response.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 2 · Did the renderer change only the intended part?

**How to read the figure.** Figure 2a is the renderer's semantic preflight: for each part it shows the
original, replacement, deletion, original part map, and replacement part
map. In a visible example, the named part should change while body, pose,
camera, and background remain fixed. A cached row with identical original
and replacement RGB pixels is not called visibly changed; it is retained for
the later exact visibility analysis. This is a pixel-operation gate; it
contains no model result.


- **Method in one line:** We displayed the renderer's saved original, one-part replacement, deletion, and part-mask outputs for all five parts; this is a pixel-operation audit with no model fitting.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: FunnyBird semantic renderer preflight showing the intended one-part replacement and deletion for every part.
show_standard('f2a','fa3f4e2dfabc210875bccd677f18971e1699432b580c78078a934b4c412add4e')


Shared across gamma: this is the identical renderer/data evidence, not six new model results. The MCBM manifest audit above verifies matching replacement bytes.


## Standard construction f2b

### Figure 2b · Do saved examples confirm the operation for every part?

**Question.** Does the accepted swap output contain a visually inspectable
original, replacement, deletion, and replacement-part map for tail, wing,
beak, foot, and eye?

**Variables and prediction.** Each row is one named part and each column is
one image role. A valid visible example changes the named part and its map
while leaving the remaining bird and scene unchanged. Across the complete
cache, 98.3% of replacement RGB images differ from their original; the
remaining 1.7% are retained for the later visibility analysis rather than
described as visibly changed. “Missing” is an error, not evidence.

**Method.** Select the first stored audit example by filename order for each
part. This is a complete five-part semantic check, not a hand-picked model
success/failure gallery.

**How to read the figure.** Compare columns within a row, then compare the
visible changed pixels with the highlighted replacement-part map. No axis or
color encodes a model score.


- **Method in one line:** We selected one accepted saved audit row per part and displayed its original, replacement, deletion, and isolated target mask; no result was estimated from these examples.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Complete five-part FunnyBird intervention audit showing original, replacement, deletion, and replacement-part map.
show_standard('f2b','0b3ed37591394ec19f5f3290aee3840b40658f00be46eccb09b6d40c2f160fdc')


Shared across gamma: this is the identical renderer/data evidence, not six new model results. The MCBM manifest audit above verifies matching replacement bytes.


## Standard construction f3

## 3 · Did the inserted pixels move the comparison toward the donor?

**Question.** Did the inserted pixels move the comparison toward the donor?

**Variables and prediction.** `response_delta = (z_donor-z_source)_cf - (z_donor-z_source)_orig`. Legacy CSV columns named `z_*` contain these post-head raw logits. Values above zero mean that replacement pixels moved the model toward the donor concept.

**Method.** Plot the complete distribution for every part and report the positive-response rate.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 3 · Did the inserted pixels move the comparison toward the donor?

**How to read the figure.** Panel A puts part on the x-axis and `response_delta` in raw-logit units on the
y-axis. The box spans the 25th--75th percentiles, the orange line is the
median, and whiskers are the 5th--95th percentiles; outliers are omitted only
from drawing. Zero means no donorward change and values above zero mean the
donor gained relative to the old source. Panel B reports the fraction above
zero, with `n` printed over each bar. Colors identify parts using the shared
FunnyBird palette. This measures response size, not whether the donor wins.
Example: a margin change from -20 before replacement to -5 afterward gives
`response_delta=+15`, although the final margin remains negative.


- **Method in one line:** For each of 5,000 paired swaps, we subtracted the original donor-minus-source margin from the counterfactual margin, then summarized those paired raw-logit changes by part; nothing was trained.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
show_standard('f3','c66e716777071168111da7b2ff7dd26523a4d549d7ee3661173aa6b6863670cc')


In [ ]:
# ALT: MCBM gamma=0: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))

PARITY_RUN.append(dict(figure='f3',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))

PARITY_RUN.append(dict(figure='f3',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))

PARITY_RUN.append(dict(figure='f3',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))

PARITY_RUN.append(dict(figure='f3',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))

PARITY_RUN.append(dict(figure='f3',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))

PARITY_RUN.append(dict(figure='f3',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f3b

## 3b — Where did each part start, and which score changed after replacement?

**Question.** Does a part finish poorly because its donor began far below
the source, because the donor rose too little, because the removed source
fell too little, or because several of these occurred together?

**Variables and exact identity.** For every swap:

`m_orig = z_donor,orig - z_source,orig`

`donor_gain = z_donor,cf - z_donor,orig`

`source_decrease = z_source,orig - z_source,cf`

`response_delta = donor_gain + source_decrease`

`m_cf = m_orig + response_delta`

**Score scale.** Every quantity here uses the post-head raw logit
`z=q(h)`, which is unbounded. This standard CBM has no MCBM gamma penalty
and no `±3` target. Notebook 03 applies the soft `±3` target to internal
`h`, not to the plotted `z`.

`m_orig` is the starting preference on the unchanged original image. It
is **not** a pure context measurement because the source part is still
visible there. Species/body context is tested separately later.

Example: the donor starts 20 units below the source, then rises by 9 while
the old source falls by 6. Total donorward response is 15, so the final
margin is `-20+9+6=-5`: the swap helped, but the source still wins.

**Method.** Average each raw-logit quantity over all 1,000 validated swaps
for each part, including both directions. Verify the exact row-wise
identity before displaying any mean.

### Figure 3b — Starting preference, donor rise, source release, response, and final result

**How to read the figure.** All panels use the same raw-logit y-axis and
part colors. Panel A below zero means the future donor starts behind.
Panels B and C above zero are the two ways replacement helps. Panel D is
their sum. Panel E above zero means the donor finally wins. Part names
identify observed outcomes, not mechanisms.


- **Method in one line:** We arithmetically decomposed every paired swap into starting margin, donor-score rise, source-score fall, total response, and final margin, verified the identity row by row, and averaged each term by part.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
show_standard('f3b','ea2f39f8bdd5fe41ffdb01bdd7656130c6225a736b72a698e819d7041f82a8c2')


In [ ]:
# ALT: MCBM gamma=0: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)

PARITY_RUN.append(dict(figure='f3b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)

PARITY_RUN.append(dict(figure='f3b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)

PARITY_RUN.append(dict(figure='f3b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)

PARITY_RUN.append(dict(figure='f3b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)

PARITY_RUN.append(dict(figure='f3b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)

PARITY_RUN.append(dict(figure='f3b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f4

## 4 · After responding, does the donor finish above the old source?

**Question.** After responding, does the donor finish above the old source?

**Variables and prediction.** The final margin is `m_cf=z_donor,cf-z_source,cf`. The primary event is `response_delta>0` with `m_cf<0`. A lower-right quadrant point means the inserted pixels had an effect but the old source still wins.

**Method.** Show final-margin distributions and the joint response/margin plane for every part.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 4 · After responding, does the donor finish above the old source?

**How to read the figure.** In the margin panel, zero separates donor wins (`m_cf>0`) from old-source wins
(`m_cf<0`). In the quadrant panel, x is donorward movement and y is the final
donor-minus-source score. The lower-right quadrant is the controlled
backwash predicate `response_delta>0 and m_cf<0`: the new pixels moved the
answer toward the donor, but the old source still finished higher. Boxes and colors use the Figure 3 definitions;
translucent points are individual swaps and the legend maps color to part.
Example: `m_cf=-5` means the old source finishes five raw-logit units above
the donor.


- **Method in one line:** We applied the predeclared row-level predicate `response_delta>0 and m_cf<0` to all 1,000 validated swaps per part and plotted the two raw quantities jointly; no threshold was learned.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
show_standard('f4','756706a0764dd5d8474f6addb659b845de363c428bfdc4c18e2ddaa8a33294f2')


In [ ]:
# ALT: MCBM gamma=0: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))

PARITY_RUN.append(dict(figure='f4',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))

PARITY_RUN.append(dict(figure='f4',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))

PARITY_RUN.append(dict(figure='f4',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))

PARITY_RUN.append(dict(figure='f4',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))

PARITY_RUN.append(dict(figure='f4',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))

PARITY_RUN.append(dict(figure='f4',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f4b

## 4b — How often does the donor win, help but still lose, or fail to move donorward?

Every validated swap is placed into exactly one outcome:

1. `m_cf > 0`: the donor concept finishes higher;
2. `m_cf <= 0 and response_delta > 0`: the new pixels help, but the old
   source concept remains higher;
3. `m_cf <= 0 and response_delta <= 0`: the source remains higher and the
   replacement does not move the comparison toward the donor.

These fractions sum to one for every part. Thus Figure 4's controlled-
backwash rate is not the donor-win rate.

Example: if 20 of 100 swaps end donor-positive, 50 move donorward but
remain source-negative, and 30 do not move donorward, the three displayed
fractions are 0.20, 0.50, and 0.30. The denominator is all 100 swaps.

### Figure 4b — Three mutually exclusive outcomes for every part

**How to read the figure.** Every panel contains all five parts and uses a
fraction from zero to one. Higher is desirable only in Panel A. Panel B
is the controlled backwash event. Panel C is a different failure: no
positive response. Both swap directions are included. Bar colors use the
shared part palette defined at the start of the notebook.


- **Method in one line:** We assigned every swap to exactly one of three predeclared outcomes from the signs of `m_cf` and `response_delta`, then divided each count by all 1,000 swaps for that part.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
show_standard('f4b','2341a3f225887aebf199db0a444ab5772b9bdd74d0efcaddc36803e3dadcc310')


In [ ]:
# ALT: MCBM gamma=0: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_RUN.append(dict(figure='f4b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_RUN.append(dict(figure='f4b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_RUN.append(dict(figure='f4b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_RUN.append(dict(figure='f4b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_RUN.append(dict(figure='f4b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird MCBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_RUN.append(dict(figure='f4b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f5

## 5 · Could opposite swap directions create the result?

**Question.** Could opposite swap directions create the result?

**Variables and prediction.** Compare forward and backward rates of `response_delta>0 and final margin<0`, together with median margins. A genuine part pattern should appear in both directions rather than cancel when pooled.

**Method.** Keep directions separate and show their denominators.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 5 · Could opposite swap directions create the result?

**How to read the figure.** Each part has separate forward (`fwd`) and backward (`bwd`) replacement
estimates, shown as unconnected circles and squares. The rate is
the fraction of rows in the lower-right quadrant from Figure 4; the printed
denominator is the number of swaps. Similar values in both directions argue
against a pooled average hiding opposite effects. A rate of 0.60 means 60%
of swaps in that direction satisfy both `response_delta>0` and `m_cf<0`.


- **Method in one line:** We split each part's 1,000 fixed swaps into its 500 forward and 500 backward replacements and recomputed the same controlled-event rate and final-margin summary independently; nothing was fitted.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
show_standard('f5','422ba05a3fb417ecbfe6cc654c516f24563aef4aead92046bf636e3f91961b08')


In [ ]:
# ALT: MCBM gamma=0: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))

PARITY_RUN.append(dict(figure='f5',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))

PARITY_RUN.append(dict(figure='f5',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))

PARITY_RUN.append(dict(figure='f5',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))

PARITY_RUN.append(dict(figure='f5',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))

PARITY_RUN.append(dict(figure='f5',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))

PARITY_RUN.append(dict(figure='f5',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f6

## 6 · How much of the result is associated with target visibility?

**Question.** How much of the result is associated with target visibility?

**Variables and prediction.** Use `pixel_count_cf` from the exact swapped-part map and the same final-margin and `response_delta>0, margin<0` definition. If visibility is sufficient, highly visible replacements should remove the part gap; a remaining gap requires another explanation.

**Method.** Use declared bins and print the number of swap rows in every bin.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 6 · How much of the result is associated with target visibility?

**How to read the figure.** The x-axis bins swaps by the number of visible pixels in the inserted target
part. Eye, wing, and foot sum both official left/right renderer-instance
colors; beak and tail each use their single official color. The original
accepted CSV's first-instance count is retained separately for audit and is
not used here. One panel shows median final raw-logit margin; the other shows the
responded-but-source-wins fraction. If visibility were the whole explanation,
sufficiently large visible parts should make margins positive and drive that
fraction near zero for every part. Point color identifies part; the table
gives the exact denominator for every nonempty bin. The companion visible-only
summary uses the same rule for all parts: `pixel_count_cf > 0`. A median
margin of +3 means the donor finishes three raw-logit units above the source.
Example: a replacement with 120 target-part pixels enters the `100--199`
bin; binning records visibility already present in the render and does not
add pixels to the image or information to the CBM.


- **Method in one line:** We grouped the same fixed swaps by the number of visible inserted-part mask pixels and recomputed median final margin and controlled-event fraction inside each declared bin; the images and CBM were unchanged.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
show_standard('f6','7300bd1a2ea661fa54bf832f85302465f9c474126219ebd2568573623f8a0395')


In [ ]:
# ALT: MCBM gamma=0: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p],lw=1.2)
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p],lw=1.2)
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))

PARITY_RUN.append(dict(figure='f6',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p],lw=1.2)
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p],lw=1.2)
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))

PARITY_RUN.append(dict(figure='f6',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p],lw=1.2)
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p],lw=1.2)
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))

PARITY_RUN.append(dict(figure='f6',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p],lw=1.2)
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p],lw=1.2)
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))

PARITY_RUN.append(dict(figure='f6',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p],lw=1.2)
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p],lw=1.2)
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))

PARITY_RUN.append(dict(figure='f6',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p],lw=1.2)
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p],lw=1.2)
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))

PARITY_RUN.append(dict(figure='f6',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f6b

## 6b · How often did the original training label conflict with visible part evidence?

**Question.** How often did the original training label conflict with visible part evidence?

**Variables and prediction.** Compare the standard and visibility-aware label views for every image used in final training (train plus validation); count positive concept labels changed to zero within each exact concept and part group. A large conflict count identifies a plausible training signal that can reward contextual prediction, but its causal effect belongs to notebook 02rl.

**Method.** Require identical ordered image/class records in both splits and allow only `attribute_label` to differ. This cell compares data labels, not Standard and RLv2 model predictions.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 6b · How often did the original training label conflict with visible part evidence?

**How to read the figure.** Each row is one exact concept. The x-axis is
`P(visibility-aware label=0 | original label=1)`: the number of original
positive training labels removed by the visibility rule divided by all
original positive labels for that concept. A value of 0.25 means 25 of 100
positive labels conflict with visible part evidence. Color identifies part.
This is a data rate, not a model probability or causal model effect.


- **Method in one line:** We joined the Standard and visibility-aware train-plus-validation label records image by image and counted only positive labels changed to zero by the visibility rule; this is a data audit, not a model comparison.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: FunnyBird training-image counts whose positive part-concept labels change under the matched visibility-aware relabeling rule.
show_standard('f6b','ab380cee73f6ca84956c2b15817f7e43f792e468ad9ec7ec82fbd389dc0e6efd')


Shared across gamma: this is the identical renderer/data evidence, not six new model results. The MCBM manifest audit above verifies matching replacement bytes.


## Standard construction f6c

## 6c · Does label conflict line up with the exact score movement it could weaken?

Figure 6b counted a training-data problem. This figure asks whether that
problem lines up with model behavior for the **same exact value**.

- `conflict_j = hidden positive training labels for j / all positive
  training labels for j`.
- Inserted-value gain is
  `mean(z_donor,cf - z_donor,orig)` over swaps inserting value `j`.
- Removed-value fall is
  `mean(z_source,orig - z_source,cf)` over swaps removing value `j`.

Example: if `tail_4` is positive 100 times during training and invisible
in 39, its conflict rate is 0.39. If its score rises by 3.4 logit units on
average when inserted, Panel A places it at `(0.39, 3.4)`. Larger circles
mean more FunnyBird species naturally carry that exact value. Tail values
are labelled because they occupy the high-conflict region.

Moving right means more contradictory positive supervision. Moving down
means a weaker response to adding or removing the named pixels. This is
an exact-value association, not a causal percentage: part, visibility,
frequency, and appearance still differ. The matched RLv2 retraining in
notebook 02rl is the causal label test.

### Figure 6c · Exact-value label conflict versus matching score response


- **Method in one line:** We joined the 26 exact-value conflict rates from matched Standard/RLv2 training records to mean score movement for swaps inserting or removing the same value; no model was fitted or retrained.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
show_standard('f6c','5636adeeb056c433835eadb6c8e1f77f343c479bae9b3ad2b03d163d25db164a')


In [ ]:
# ALT: MCBM gamma=0: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
CONFLICT_RESPONSE=conflict_response_table(S,CONFLICT)
required={"part","value","conflict_rate","species_support",
          "mean_donor_gain","mean_source_decrease","donor_rows","source_rows"}
if required-set(CONFLICT_RESPONSE):
    raise RuntimeError(f"conflict-response table missing {sorted(required-set(CONFLICT_RESPONSE))}")
if len(CONFLICT_RESPONSE)!=26 or set(CONFLICT_RESPONSE.part)!=set(ORDER):
    raise RuntimeError("conflict-response table is not the complete 26-value FunnyBird population")
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
panels=[("mean_donor_gain","mean rise in inserted-value raw logit",
         "A · Conflict of the inserted value"),
        ("mean_source_decrease","mean fall in removed-value raw logit",
         "B · Conflict of the removed value")]
for ax,(column,ylabel,title) in zip(axes,panels):
    for part in ORDER:
        group=CONFLICT_RESPONSE[CONFLICT_RESPONSE.part==part]
        ax.scatter(group.conflict_rate,group[column],
                   s=30+7*group.species_support,color=COLORS[part],
                   label=part,alpha=.85)
        if part=="tail":
            for row in group.itertuples():
                ax.annotate(f"tail_{row.value}",(row.conflict_rate,getattr(row,column)),
                            xytext=(3,3),textcoords="offset points",fontsize=7)
    ax.set_xlabel("positive-label / invisible-mask conflict fraction")
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].legend(title="part",fontsize=8)
fig.suptitle("Figure 6c · Does label conflict match the score movement it could weaken?")
plt.tight_layout(); plt.show()
display(CONFLICT_RESPONSE.sort_values(["part","value"]).round(4))

PARITY_RUN.append(dict(figure='f6c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
CONFLICT_RESPONSE=conflict_response_table(S,CONFLICT)
required={"part","value","conflict_rate","species_support",
          "mean_donor_gain","mean_source_decrease","donor_rows","source_rows"}
if required-set(CONFLICT_RESPONSE):
    raise RuntimeError(f"conflict-response table missing {sorted(required-set(CONFLICT_RESPONSE))}")
if len(CONFLICT_RESPONSE)!=26 or set(CONFLICT_RESPONSE.part)!=set(ORDER):
    raise RuntimeError("conflict-response table is not the complete 26-value FunnyBird population")
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
panels=[("mean_donor_gain","mean rise in inserted-value raw logit",
         "A · Conflict of the inserted value"),
        ("mean_source_decrease","mean fall in removed-value raw logit",
         "B · Conflict of the removed value")]
for ax,(column,ylabel,title) in zip(axes,panels):
    for part in ORDER:
        group=CONFLICT_RESPONSE[CONFLICT_RESPONSE.part==part]
        ax.scatter(group.conflict_rate,group[column],
                   s=30+7*group.species_support,color=COLORS[part],
                   label=part,alpha=.85)
        if part=="tail":
            for row in group.itertuples():
                ax.annotate(f"tail_{row.value}",(row.conflict_rate,getattr(row,column)),
                            xytext=(3,3),textcoords="offset points",fontsize=7)
    ax.set_xlabel("positive-label / invisible-mask conflict fraction")
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].legend(title="part",fontsize=8)
fig.suptitle("Figure 6c · Does label conflict match the score movement it could weaken?")
plt.tight_layout(); plt.show()
display(CONFLICT_RESPONSE.sort_values(["part","value"]).round(4))

PARITY_RUN.append(dict(figure='f6c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
CONFLICT_RESPONSE=conflict_response_table(S,CONFLICT)
required={"part","value","conflict_rate","species_support",
          "mean_donor_gain","mean_source_decrease","donor_rows","source_rows"}
if required-set(CONFLICT_RESPONSE):
    raise RuntimeError(f"conflict-response table missing {sorted(required-set(CONFLICT_RESPONSE))}")
if len(CONFLICT_RESPONSE)!=26 or set(CONFLICT_RESPONSE.part)!=set(ORDER):
    raise RuntimeError("conflict-response table is not the complete 26-value FunnyBird population")
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
panels=[("mean_donor_gain","mean rise in inserted-value raw logit",
         "A · Conflict of the inserted value"),
        ("mean_source_decrease","mean fall in removed-value raw logit",
         "B · Conflict of the removed value")]
for ax,(column,ylabel,title) in zip(axes,panels):
    for part in ORDER:
        group=CONFLICT_RESPONSE[CONFLICT_RESPONSE.part==part]
        ax.scatter(group.conflict_rate,group[column],
                   s=30+7*group.species_support,color=COLORS[part],
                   label=part,alpha=.85)
        if part=="tail":
            for row in group.itertuples():
                ax.annotate(f"tail_{row.value}",(row.conflict_rate,getattr(row,column)),
                            xytext=(3,3),textcoords="offset points",fontsize=7)
    ax.set_xlabel("positive-label / invisible-mask conflict fraction")
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].legend(title="part",fontsize=8)
fig.suptitle("Figure 6c · Does label conflict match the score movement it could weaken?")
plt.tight_layout(); plt.show()
display(CONFLICT_RESPONSE.sort_values(["part","value"]).round(4))

PARITY_RUN.append(dict(figure='f6c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
CONFLICT_RESPONSE=conflict_response_table(S,CONFLICT)
required={"part","value","conflict_rate","species_support",
          "mean_donor_gain","mean_source_decrease","donor_rows","source_rows"}
if required-set(CONFLICT_RESPONSE):
    raise RuntimeError(f"conflict-response table missing {sorted(required-set(CONFLICT_RESPONSE))}")
if len(CONFLICT_RESPONSE)!=26 or set(CONFLICT_RESPONSE.part)!=set(ORDER):
    raise RuntimeError("conflict-response table is not the complete 26-value FunnyBird population")
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
panels=[("mean_donor_gain","mean rise in inserted-value raw logit",
         "A · Conflict of the inserted value"),
        ("mean_source_decrease","mean fall in removed-value raw logit",
         "B · Conflict of the removed value")]
for ax,(column,ylabel,title) in zip(axes,panels):
    for part in ORDER:
        group=CONFLICT_RESPONSE[CONFLICT_RESPONSE.part==part]
        ax.scatter(group.conflict_rate,group[column],
                   s=30+7*group.species_support,color=COLORS[part],
                   label=part,alpha=.85)
        if part=="tail":
            for row in group.itertuples():
                ax.annotate(f"tail_{row.value}",(row.conflict_rate,getattr(row,column)),
                            xytext=(3,3),textcoords="offset points",fontsize=7)
    ax.set_xlabel("positive-label / invisible-mask conflict fraction")
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].legend(title="part",fontsize=8)
fig.suptitle("Figure 6c · Does label conflict match the score movement it could weaken?")
plt.tight_layout(); plt.show()
display(CONFLICT_RESPONSE.sort_values(["part","value"]).round(4))

PARITY_RUN.append(dict(figure='f6c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
CONFLICT_RESPONSE=conflict_response_table(S,CONFLICT)
required={"part","value","conflict_rate","species_support",
          "mean_donor_gain","mean_source_decrease","donor_rows","source_rows"}
if required-set(CONFLICT_RESPONSE):
    raise RuntimeError(f"conflict-response table missing {sorted(required-set(CONFLICT_RESPONSE))}")
if len(CONFLICT_RESPONSE)!=26 or set(CONFLICT_RESPONSE.part)!=set(ORDER):
    raise RuntimeError("conflict-response table is not the complete 26-value FunnyBird population")
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
panels=[("mean_donor_gain","mean rise in inserted-value raw logit",
         "A · Conflict of the inserted value"),
        ("mean_source_decrease","mean fall in removed-value raw logit",
         "B · Conflict of the removed value")]
for ax,(column,ylabel,title) in zip(axes,panels):
    for part in ORDER:
        group=CONFLICT_RESPONSE[CONFLICT_RESPONSE.part==part]
        ax.scatter(group.conflict_rate,group[column],
                   s=30+7*group.species_support,color=COLORS[part],
                   label=part,alpha=.85)
        if part=="tail":
            for row in group.itertuples():
                ax.annotate(f"tail_{row.value}",(row.conflict_rate,getattr(row,column)),
                            xytext=(3,3),textcoords="offset points",fontsize=7)
    ax.set_xlabel("positive-label / invisible-mask conflict fraction")
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].legend(title="part",fontsize=8)
fig.suptitle("Figure 6c · Does label conflict match the score movement it could weaken?")
plt.tight_layout(); plt.show()
display(CONFLICT_RESPONSE.sort_values(["part","value"]).round(4))

PARITY_RUN.append(dict(figure='f6c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Two exact-value scatterplots relating training label/mask conflict to the matching inserted-score rise and removed-score fall.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
CONFLICT_RESPONSE=conflict_response_table(S,CONFLICT)
required={"part","value","conflict_rate","species_support",
          "mean_donor_gain","mean_source_decrease","donor_rows","source_rows"}
if required-set(CONFLICT_RESPONSE):
    raise RuntimeError(f"conflict-response table missing {sorted(required-set(CONFLICT_RESPONSE))}")
if len(CONFLICT_RESPONSE)!=26 or set(CONFLICT_RESPONSE.part)!=set(ORDER):
    raise RuntimeError("conflict-response table is not the complete 26-value FunnyBird population")
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
panels=[("mean_donor_gain","mean rise in inserted-value raw logit",
         "A · Conflict of the inserted value"),
        ("mean_source_decrease","mean fall in removed-value raw logit",
         "B · Conflict of the removed value")]
for ax,(column,ylabel,title) in zip(axes,panels):
    for part in ORDER:
        group=CONFLICT_RESPONSE[CONFLICT_RESPONSE.part==part]
        ax.scatter(group.conflict_rate,group[column],
                   s=30+7*group.species_support,color=COLORS[part],
                   label=part,alpha=.85)
        if part=="tail":
            for row in group.itertuples():
                ax.annotate(f"tail_{row.value}",(row.conflict_rate,getattr(row,column)),
                            xytext=(3,3),textcoords="offset points",fontsize=7)
    ax.set_xlabel("positive-label / invisible-mask conflict fraction")
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].legend(title="part",fontsize=8)
fig.suptitle("Figure 6c · Does label conflict match the score movement it could weaken?")
plt.tight_layout(); plt.show()
display(CONFLICT_RESPONSE.sort_values(["part","value"]).round(4))

PARITY_RUN.append(dict(figure='f6c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f7

## 7 · Do exact source and donor values explain the failures?

**Question.** Do exact source and donor values explain the failures?

**Variables and prediction.** For every part, compare the inserted donor value with the concept value that has the largest post-swap raw score. A clean diagonal means exact visual values are distinguished; recurring bright columns indicate default answers.

**Method.** Display all parts and all values with row-normalized counts.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 7 · Do exact source and donor values explain the failures?

**How to read the figure.** Each heatmap row is the value actually inserted and each column is the value
with the largest post-swap raw logit. A bright diagonal means the model names
the inserted value; bright off-diagonal cells show systematic confusion.
Every FunnyBird part and every value is included. The lower row gives the
final-margin distribution for the same inserted values, with the number of
swaps printed above each box. Thus recognition and retained-source margin are
visible together rather than inferred from a diagonal rate alone. A diagonal
value of 0.80 means the inserted value is highest in 80% of that row's swaps.


- **Method in one line:** For each swapped part, we took the highest post-swap raw logit within that part block, cross-tabulated it against the inserted exact value, normalized each inserted-value row, and retained every swap.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
show_standard('f7','38446cadc8f00c4abe86c70b4faedd9f50b740e391e20e52950548d0055ac86e')


In [ ]:
# ALT: MCBM gamma=0: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))

PARITY_CACHE[GAMMA]['diag']=diag
PARITY_RUN.append(dict(figure='f7',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))

PARITY_CACHE[GAMMA]['diag']=diag
PARITY_RUN.append(dict(figure='f7',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))

PARITY_CACHE[GAMMA]['diag']=diag
PARITY_RUN.append(dict(figure='f7',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))

PARITY_CACHE[GAMMA]['diag']=diag
PARITY_RUN.append(dict(figure='f7',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))

PARITY_CACHE[GAMMA]['diag']=diag
PARITY_RUN.append(dict(figure='f7',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))

PARITY_CACHE[GAMMA]['diag']=diag
PARITY_RUN.append(dict(figure='f7',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f7a

### Figure 7a · What do the nine tail values actually look like?

The confusion matrix says which values the model mixes up, but it cannot
tell a reader whether two tail shapes look similar. This compact visual
audit shows one large, clearly visible accepted replacement for each of
the nine inserted tail values. Only pixels inside the renderer's tail
mask are retained, so body shape, pose, and background do not dominate
the comparison. The examples are selected mechanically by largest tail
mask area, not by model success or failure.

This is a human-readable visual check, not a new backwash measurement.
Apparent similarity can motivate a hypothesis. In the displayed examples,
the repeated-outline families are approximately `0/3/6`, `1/4/7`, and
`2/5/8`, with color changing inside each family. The confusion matrix can
then be checked for errors within or across those visible families, but the
crops alone do not establish a cause; the controlled margins remain the
quantitative evidence.


- **Method in one line:** We selected the largest accepted tail mask for each exact inserted tail value and displayed only its masked RGB pixels; no score, classifier, or outcome was used for selection.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Nine isolated renderer-tail crops, one mechanically selected large-mask example per exact tail value.
show_standard('f7a','458650adef91c5ebe94824048bde3d30b6ca8efb809d1e331b46c1a4d0cae212')


Shared across gamma: this is the identical renderer/data evidence, not six new model results. The MCBM manifest audit above verifies matching replacement bytes.


## Standard construction f7b

## 7b · Are difficult values simply rare or drawn from a larger alternative set?

**Question.** Are difficult values simply rare or drawn from a larger alternative set?

**Variables and prediction.** For every exact donor value, compare its species support with all three mutually exclusive outcomes from Figure 4b; also report the total number of alternatives for its part. If rarity organizes the result, lower-support values should systematically win less or fail more. A mixed pattern rejects support as a sufficient explanation.

**Method.** Label every exact value, print its swap-row denominator, and verify that its three outcome fractions sum to one.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 7b · Are difficult values simply rare or drawn from a larger alternative set?

**How to read the figure.** Each labelled point is one exact donor value. The x-axis is its species
support: the number of the 50 FunnyBird species that naturally carry that
value in an unmodified bird. It is not an image count or swap count. The
count comes from the renderer's species-to-part-value definition: if six
species ordinarily have donor value 2, its support is 6 even when the swap
table contains hundreds of value-2 rows. The
three panels partition all swaps into donor wins, donorward movement that
remains source-negative, and no donorward movement while source-negative.
The three fractions sum to one within each value. A consistent relationship
with support would make rarity a plausible organizer. The number of
alternatives is reported but cannot be cleanly separated with only five
parts.


- **Method in one line:** We counted how many of the 50 species naturally carry each donor value, then grouped all swap rows for that value into the three exhaustive Figure 4b outcomes; no correlation model was fitted.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
show_standard('f7b','ff5308d5b5a6a77c8142d537b9d558a5af2bd41830172224efe1b8f06894d8a1')


In [ ]:
# ALT: MCBM gamma=0: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
support_correlations=[]
for part,group in VS.groupby("part"):
    support_correlations.append({
        "part":part,"exact_values":len(group),
        "support_vs_donor_wins_spearman":group.species_support.corr(group.donor_wins_rate,method="spearman"),
        "support_vs_controlled_event_spearman":group.species_support.corr(group.responded_but_source_wins_rate,method="spearman"),
    })
SUPPORT_CORRELATIONS=pd.DataFrame(support_correlations).set_index("part").reindex(ORDER)
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))
print("Within-part rank associations (descriptive; very few exact values per part):")
display(SUPPORT_CORRELATIONS.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_CACHE[GAMMA]['VS']=VS
PARITY_RUN.append(dict(figure='f7b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
support_correlations=[]
for part,group in VS.groupby("part"):
    support_correlations.append({
        "part":part,"exact_values":len(group),
        "support_vs_donor_wins_spearman":group.species_support.corr(group.donor_wins_rate,method="spearman"),
        "support_vs_controlled_event_spearman":group.species_support.corr(group.responded_but_source_wins_rate,method="spearman"),
    })
SUPPORT_CORRELATIONS=pd.DataFrame(support_correlations).set_index("part").reindex(ORDER)
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))
print("Within-part rank associations (descriptive; very few exact values per part):")
display(SUPPORT_CORRELATIONS.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_CACHE[GAMMA]['VS']=VS
PARITY_RUN.append(dict(figure='f7b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
support_correlations=[]
for part,group in VS.groupby("part"):
    support_correlations.append({
        "part":part,"exact_values":len(group),
        "support_vs_donor_wins_spearman":group.species_support.corr(group.donor_wins_rate,method="spearman"),
        "support_vs_controlled_event_spearman":group.species_support.corr(group.responded_but_source_wins_rate,method="spearman"),
    })
SUPPORT_CORRELATIONS=pd.DataFrame(support_correlations).set_index("part").reindex(ORDER)
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))
print("Within-part rank associations (descriptive; very few exact values per part):")
display(SUPPORT_CORRELATIONS.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_CACHE[GAMMA]['VS']=VS
PARITY_RUN.append(dict(figure='f7b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
support_correlations=[]
for part,group in VS.groupby("part"):
    support_correlations.append({
        "part":part,"exact_values":len(group),
        "support_vs_donor_wins_spearman":group.species_support.corr(group.donor_wins_rate,method="spearman"),
        "support_vs_controlled_event_spearman":group.species_support.corr(group.responded_but_source_wins_rate,method="spearman"),
    })
SUPPORT_CORRELATIONS=pd.DataFrame(support_correlations).set_index("part").reindex(ORDER)
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))
print("Within-part rank associations (descriptive; very few exact values per part):")
display(SUPPORT_CORRELATIONS.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_CACHE[GAMMA]['VS']=VS
PARITY_RUN.append(dict(figure='f7b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
support_correlations=[]
for part,group in VS.groupby("part"):
    support_correlations.append({
        "part":part,"exact_values":len(group),
        "support_vs_donor_wins_spearman":group.species_support.corr(group.donor_wins_rate,method="spearman"),
        "support_vs_controlled_event_spearman":group.species_support.corr(group.responded_but_source_wins_rate,method="spearman"),
    })
SUPPORT_CORRELATIONS=pd.DataFrame(support_correlations).set_index("part").reindex(ORDER)
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))
print("Within-part rank associations (descriptive; very few exact values per part):")
display(SUPPORT_CORRELATIONS.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_CACHE[GAMMA]['VS']=VS
PARITY_RUN.append(dict(figure='f7b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
support_correlations=[]
for part,group in VS.groupby("part"):
    support_correlations.append({
        "part":part,"exact_values":len(group),
        "support_vs_donor_wins_spearman":group.species_support.corr(group.donor_wins_rate,method="spearman"),
        "support_vs_controlled_event_spearman":group.species_support.corr(group.responded_but_source_wins_rate,method="spearman"),
    })
SUPPORT_CORRELATIONS=pd.DataFrame(support_correlations).set_index("part").reindex(ORDER)
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))
print("Within-part rank associations (descriptive; very few exact values per part):")
display(SUPPORT_CORRELATIONS.round(3))

print('Exact final-margin ties (not strict source wins):',int((S.m_cf==0).sum()))

PARITY_CACHE[GAMMA]['VS']=VS
PARITY_RUN.append(dict(figure='f7b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f7c

## 7c · Does rarity help explain why some concept heads start farther below zero?

**Question.** Figure 7b showed that support alone cannot explain why tail
fails while similarly rare wing values usually succeed. It may still
explain a different link: the ordinary score scale learned by each head.

**Definitions.** `support_j` is the number of the 50 species naturally
carrying exact value `j`. `absent_mean_j` is the average raw score `z_j`
among the the printed number of ordinary held-out images whose official label is zero.
More negative means the head normally pushes that absent value farther
below the zero decision boundary.

**Worked example.** If only two species use `tail_7`, support is 2. If
three ordinary images without tail 7 score `-8,-6,-7`, the absent mean is
`(-8-6-7)/3=-7`. A donor swap must lift that deeply negative score farther
than a head whose absent mean is `-3`.

**Prediction.** If rarity contributes to the starting deficit, lower
support should accompany a more negative absent mean. Rarity was not
randomized; appearance, conflict, and part identity remain alternatives.

### Figure 7c · Species support versus ordinary absent raw score

**How to read it.** Each labelled point is one exact value. Right means
more species carry it. Up means its output is less negative when absent.
The line is a descriptive least-squares summary, not a causal model or
uncertainty interval. Tables print every value and rank correlations.


- **Method in one line:** For each exact value we counted its natural species support and averaged the frozen Standard CBM raw logit only over ordinary held-out images where that value's official label is zero; the fitted line is descriptive.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
show_standard('f7c','ccf045a1e97b14bc22e5fce1d0c3656f5c6cdd9800c3a4f7ebfdfbb24355340d')


In [ ]:
# ALT: MCBM gamma=0: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
VS=PARITY_CACHE[GAMMA]['VS']
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
absent_rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    labels=c_saved[:,j].astype(int); absent=z_saved[labels==0,j]
    part=CONCEPT_PART[name]; local_value=j-SPANS[part][0]
    support=int(VS.query("part==@part and var_donor==@local_value").species_support.iloc[0])
    absent_rows.append({"part":part,"concept":name,"value":local_value,
                        "species_support":support,"N_absent":len(absent),
                        "absent_mean":absent.mean(),"absent_SD":absent.std(ddof=1)})
RARITY_CALIBRATION=pd.DataFrame(absent_rows)
fig,ax=plt.subplots(figsize=(10,6))
for part in ORDER:
    d=RARITY_CALIBRATION.query("part==@part")
    ax.scatter(d.species_support,d.absent_mean,s=55,color=COLORS[part],label=part)
    for row in d.itertuples():
        ax.annotate(f"{part[0]}{row.value}",(row.species_support,row.absent_mean),
                    xytext=(4,4),textcoords="offset points",fontsize=7)
x=RARITY_CALIBRATION.species_support.to_numpy(float)
y=RARITY_CALIBRATION.absent_mean.to_numpy(float)
slope,intercept=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
ax.plot(grid,intercept+slope*grid,color="#333333",lw=1.3,label="all-value linear summary")
ax.axhline(0,color="black",lw=.8); ax.set_xlabel("species support (of 50 species)")
ax.set_ylabel("mean raw score when the exact value is absent")
ax.set_title("Figure 7c · Do rarer values have deeper ordinary absent baselines?")
ax.legend(); plt.tight_layout(); plt.show()
display(RARITY_CALIBRATION.sort_values(["part","value"]).round(3))
tail_rows=RARITY_CALIBRATION.query("part=='tail'")
display(pd.DataFrame([
    {"population":"all 26 exact values","n_values":len(RARITY_CALIBRATION),
     "Spearman_support_vs_absent_mean":RARITY_CALIBRATION.species_support.corr(RARITY_CALIBRATION.absent_mean,method="spearman")},
    {"population":"nine tail values","n_values":len(tail_rows),
     "Spearman_support_vs_absent_mean":tail_rows.species_support.corr(tail_rows.absent_mean,method="spearman")},
]).round(3))

PARITY_RUN.append(dict(figure='f7c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
VS=PARITY_CACHE[GAMMA]['VS']
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
absent_rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    labels=c_saved[:,j].astype(int); absent=z_saved[labels==0,j]
    part=CONCEPT_PART[name]; local_value=j-SPANS[part][0]
    support=int(VS.query("part==@part and var_donor==@local_value").species_support.iloc[0])
    absent_rows.append({"part":part,"concept":name,"value":local_value,
                        "species_support":support,"N_absent":len(absent),
                        "absent_mean":absent.mean(),"absent_SD":absent.std(ddof=1)})
RARITY_CALIBRATION=pd.DataFrame(absent_rows)
fig,ax=plt.subplots(figsize=(10,6))
for part in ORDER:
    d=RARITY_CALIBRATION.query("part==@part")
    ax.scatter(d.species_support,d.absent_mean,s=55,color=COLORS[part],label=part)
    for row in d.itertuples():
        ax.annotate(f"{part[0]}{row.value}",(row.species_support,row.absent_mean),
                    xytext=(4,4),textcoords="offset points",fontsize=7)
x=RARITY_CALIBRATION.species_support.to_numpy(float)
y=RARITY_CALIBRATION.absent_mean.to_numpy(float)
slope,intercept=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
ax.plot(grid,intercept+slope*grid,color="#333333",lw=1.3,label="all-value linear summary")
ax.axhline(0,color="black",lw=.8); ax.set_xlabel("species support (of 50 species)")
ax.set_ylabel("mean raw score when the exact value is absent")
ax.set_title("Figure 7c · Do rarer values have deeper ordinary absent baselines?")
ax.legend(); plt.tight_layout(); plt.show()
display(RARITY_CALIBRATION.sort_values(["part","value"]).round(3))
tail_rows=RARITY_CALIBRATION.query("part=='tail'")
display(pd.DataFrame([
    {"population":"all 26 exact values","n_values":len(RARITY_CALIBRATION),
     "Spearman_support_vs_absent_mean":RARITY_CALIBRATION.species_support.corr(RARITY_CALIBRATION.absent_mean,method="spearman")},
    {"population":"nine tail values","n_values":len(tail_rows),
     "Spearman_support_vs_absent_mean":tail_rows.species_support.corr(tail_rows.absent_mean,method="spearman")},
]).round(3))

PARITY_RUN.append(dict(figure='f7c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
VS=PARITY_CACHE[GAMMA]['VS']
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
absent_rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    labels=c_saved[:,j].astype(int); absent=z_saved[labels==0,j]
    part=CONCEPT_PART[name]; local_value=j-SPANS[part][0]
    support=int(VS.query("part==@part and var_donor==@local_value").species_support.iloc[0])
    absent_rows.append({"part":part,"concept":name,"value":local_value,
                        "species_support":support,"N_absent":len(absent),
                        "absent_mean":absent.mean(),"absent_SD":absent.std(ddof=1)})
RARITY_CALIBRATION=pd.DataFrame(absent_rows)
fig,ax=plt.subplots(figsize=(10,6))
for part in ORDER:
    d=RARITY_CALIBRATION.query("part==@part")
    ax.scatter(d.species_support,d.absent_mean,s=55,color=COLORS[part],label=part)
    for row in d.itertuples():
        ax.annotate(f"{part[0]}{row.value}",(row.species_support,row.absent_mean),
                    xytext=(4,4),textcoords="offset points",fontsize=7)
x=RARITY_CALIBRATION.species_support.to_numpy(float)
y=RARITY_CALIBRATION.absent_mean.to_numpy(float)
slope,intercept=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
ax.plot(grid,intercept+slope*grid,color="#333333",lw=1.3,label="all-value linear summary")
ax.axhline(0,color="black",lw=.8); ax.set_xlabel("species support (of 50 species)")
ax.set_ylabel("mean raw score when the exact value is absent")
ax.set_title("Figure 7c · Do rarer values have deeper ordinary absent baselines?")
ax.legend(); plt.tight_layout(); plt.show()
display(RARITY_CALIBRATION.sort_values(["part","value"]).round(3))
tail_rows=RARITY_CALIBRATION.query("part=='tail'")
display(pd.DataFrame([
    {"population":"all 26 exact values","n_values":len(RARITY_CALIBRATION),
     "Spearman_support_vs_absent_mean":RARITY_CALIBRATION.species_support.corr(RARITY_CALIBRATION.absent_mean,method="spearman")},
    {"population":"nine tail values","n_values":len(tail_rows),
     "Spearman_support_vs_absent_mean":tail_rows.species_support.corr(tail_rows.absent_mean,method="spearman")},
]).round(3))

PARITY_RUN.append(dict(figure='f7c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
VS=PARITY_CACHE[GAMMA]['VS']
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
absent_rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    labels=c_saved[:,j].astype(int); absent=z_saved[labels==0,j]
    part=CONCEPT_PART[name]; local_value=j-SPANS[part][0]
    support=int(VS.query("part==@part and var_donor==@local_value").species_support.iloc[0])
    absent_rows.append({"part":part,"concept":name,"value":local_value,
                        "species_support":support,"N_absent":len(absent),
                        "absent_mean":absent.mean(),"absent_SD":absent.std(ddof=1)})
RARITY_CALIBRATION=pd.DataFrame(absent_rows)
fig,ax=plt.subplots(figsize=(10,6))
for part in ORDER:
    d=RARITY_CALIBRATION.query("part==@part")
    ax.scatter(d.species_support,d.absent_mean,s=55,color=COLORS[part],label=part)
    for row in d.itertuples():
        ax.annotate(f"{part[0]}{row.value}",(row.species_support,row.absent_mean),
                    xytext=(4,4),textcoords="offset points",fontsize=7)
x=RARITY_CALIBRATION.species_support.to_numpy(float)
y=RARITY_CALIBRATION.absent_mean.to_numpy(float)
slope,intercept=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
ax.plot(grid,intercept+slope*grid,color="#333333",lw=1.3,label="all-value linear summary")
ax.axhline(0,color="black",lw=.8); ax.set_xlabel("species support (of 50 species)")
ax.set_ylabel("mean raw score when the exact value is absent")
ax.set_title("Figure 7c · Do rarer values have deeper ordinary absent baselines?")
ax.legend(); plt.tight_layout(); plt.show()
display(RARITY_CALIBRATION.sort_values(["part","value"]).round(3))
tail_rows=RARITY_CALIBRATION.query("part=='tail'")
display(pd.DataFrame([
    {"population":"all 26 exact values","n_values":len(RARITY_CALIBRATION),
     "Spearman_support_vs_absent_mean":RARITY_CALIBRATION.species_support.corr(RARITY_CALIBRATION.absent_mean,method="spearman")},
    {"population":"nine tail values","n_values":len(tail_rows),
     "Spearman_support_vs_absent_mean":tail_rows.species_support.corr(tail_rows.absent_mean,method="spearman")},
]).round(3))

PARITY_RUN.append(dict(figure='f7c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
VS=PARITY_CACHE[GAMMA]['VS']
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
absent_rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    labels=c_saved[:,j].astype(int); absent=z_saved[labels==0,j]
    part=CONCEPT_PART[name]; local_value=j-SPANS[part][0]
    support=int(VS.query("part==@part and var_donor==@local_value").species_support.iloc[0])
    absent_rows.append({"part":part,"concept":name,"value":local_value,
                        "species_support":support,"N_absent":len(absent),
                        "absent_mean":absent.mean(),"absent_SD":absent.std(ddof=1)})
RARITY_CALIBRATION=pd.DataFrame(absent_rows)
fig,ax=plt.subplots(figsize=(10,6))
for part in ORDER:
    d=RARITY_CALIBRATION.query("part==@part")
    ax.scatter(d.species_support,d.absent_mean,s=55,color=COLORS[part],label=part)
    for row in d.itertuples():
        ax.annotate(f"{part[0]}{row.value}",(row.species_support,row.absent_mean),
                    xytext=(4,4),textcoords="offset points",fontsize=7)
x=RARITY_CALIBRATION.species_support.to_numpy(float)
y=RARITY_CALIBRATION.absent_mean.to_numpy(float)
slope,intercept=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
ax.plot(grid,intercept+slope*grid,color="#333333",lw=1.3,label="all-value linear summary")
ax.axhline(0,color="black",lw=.8); ax.set_xlabel("species support (of 50 species)")
ax.set_ylabel("mean raw score when the exact value is absent")
ax.set_title("Figure 7c · Do rarer values have deeper ordinary absent baselines?")
ax.legend(); plt.tight_layout(); plt.show()
display(RARITY_CALIBRATION.sort_values(["part","value"]).round(3))
tail_rows=RARITY_CALIBRATION.query("part=='tail'")
display(pd.DataFrame([
    {"population":"all 26 exact values","n_values":len(RARITY_CALIBRATION),
     "Spearman_support_vs_absent_mean":RARITY_CALIBRATION.species_support.corr(RARITY_CALIBRATION.absent_mean,method="spearman")},
    {"population":"nine tail values","n_values":len(tail_rows),
     "Spearman_support_vs_absent_mean":tail_rows.species_support.corr(tail_rows.absent_mean,method="spearman")},
]).round(3))

PARITY_RUN.append(dict(figure='f7c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
VS=PARITY_CACHE[GAMMA]['VS']
# ALT: Labelled scatter of species support against the ordinary absent raw-logit mean for all 26 exact FunnyBird values.
absent_rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    labels=c_saved[:,j].astype(int); absent=z_saved[labels==0,j]
    part=CONCEPT_PART[name]; local_value=j-SPANS[part][0]
    support=int(VS.query("part==@part and var_donor==@local_value").species_support.iloc[0])
    absent_rows.append({"part":part,"concept":name,"value":local_value,
                        "species_support":support,"N_absent":len(absent),
                        "absent_mean":absent.mean(),"absent_SD":absent.std(ddof=1)})
RARITY_CALIBRATION=pd.DataFrame(absent_rows)
fig,ax=plt.subplots(figsize=(10,6))
for part in ORDER:
    d=RARITY_CALIBRATION.query("part==@part")
    ax.scatter(d.species_support,d.absent_mean,s=55,color=COLORS[part],label=part)
    for row in d.itertuples():
        ax.annotate(f"{part[0]}{row.value}",(row.species_support,row.absent_mean),
                    xytext=(4,4),textcoords="offset points",fontsize=7)
x=RARITY_CALIBRATION.species_support.to_numpy(float)
y=RARITY_CALIBRATION.absent_mean.to_numpy(float)
slope,intercept=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
ax.plot(grid,intercept+slope*grid,color="#333333",lw=1.3,label="all-value linear summary")
ax.axhline(0,color="black",lw=.8); ax.set_xlabel("species support (of 50 species)")
ax.set_ylabel("mean raw score when the exact value is absent")
ax.set_title("Figure 7c · Do rarer values have deeper ordinary absent baselines?")
ax.legend(); plt.tight_layout(); plt.show()
display(RARITY_CALIBRATION.sort_values(["part","value"]).round(3))
tail_rows=RARITY_CALIBRATION.query("part=='tail'")
display(pd.DataFrame([
    {"population":"all 26 exact values","n_values":len(RARITY_CALIBRATION),
     "Spearman_support_vs_absent_mean":RARITY_CALIBRATION.species_support.corr(RARITY_CALIBRATION.absent_mean,method="spearman")},
    {"population":"nine tail values","n_values":len(tail_rows),
     "Spearman_support_vs_absent_mean":tail_rows.species_support.corr(tail_rows.absent_mean,method="spearman")},
]).round(3))

PARITY_RUN.append(dict(figure='f7c',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f8

## 8 · Does source species organize the remaining error after exact values?

**Question.** If two birds receive the same exact replacement, do their
source species still accompany systematically different final margins?

**Why exact-pair centering is needed.** A species could appear resistant
merely because it happens to receive difficult replacements. We first
compare each row only with swaps having the same part, old exact value,
and inserted exact value. This removes that composition difference before
species are summarized.

**Complete procedure.**

```text
for each swap row:
    exact_pair = (part, source_value, donor_value)

for each exact_pair:
    pair_mean = average final margin across all its rows

for each row:
    row_residual = row final margin - its pair_mean

for each (part, source_species):
    species_residual = average row_residual

display species only when it has at least five rows
```

**Numerical example.** Suppose red-tail to blue-tail margins are `-5`
and `-3` for Species A and `+1` and `+3` for Species B. Their exact-pair
mean is `-1`. The residuals are therefore `-4,-2,+2,+4`: they average to
zero over the pair, but Species A averages `-3` and Species B averages
`+3`. Species A is more source-retaining than the exact-pair average.

**Prediction and limit.** Persistent species residuals support an
unchanged-body/species association after exact values. They do not prove
that species causes the difference because species remains bundled with
body shape, pose tendencies, visibility, and other renderer properties.

**What the color does and does not explain.** A blue cell means that this
species/part combination finished more source-favouring than other rows
receiving the same old-to-new value replacement; red means more
donor-favouring. Color does not itself say *why*. The table printed below
the heatmap therefore shows, for the most extreme cells, the average
starting margin, donorward movement, final margin, visible-pixel count,
and exact-pair-centred residual. Frequency of the old/new values has
already been controlled by exact-pair centering. Explaining the remaining
color would require independently measured body shape, pose, mask geometry,
or another context variable—not merely reusing the species name.

### Figure 8 · Source-species residual after exact source/donor values

**How to read the figure.** Every row keeps one source-species identity
and every column keeps one replaced part. Blue cells are more
source-retaining than other swaps with the same exact source and donor
values; red cells are more donor-receptive; white is the exact-pair
average. Blank cells lack five eligible rows. Reading across one row asks
whether that unchanged bird context accompanies similar or different
residuals for several parts; no correlation threshold is imposed.


- **Method in one line:** We subtracted each ordered `(part, source value, donor value)` pair's pooled mean margin and averaged the remaining residuals by unchanged source species; for the most extreme cells we also print their starting margin, donorward movement, final margin, visibility, and strongest exact old-to-new transitions. This is descriptive centering, not a fitted causal model.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
show_standard('f8','9d405b5043e5d1bdb2741bd0b4d582aa59f0dead9f810e3e242b16113f9e00ae')


In [ ]:
# ALT: MCBM gamma=0: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_transition=(R.groupby(["part","sid_src"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
transition_detail=(R.groupby(["part","sid_src","var_src","var_donor"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
species_extremes=pd.concat([
    pd.concat([group.nsmallest(2,'exact_pair_centered_residual'),
               group.nlargest(2,'exact_pair_centered_residual')])
    for _,group in species_transition.groupby('part')
]).sort_values(['part','exact_pair_centered_residual'])
extreme_keys=species_extremes[["part","sid_src"]].drop_duplicates()
extreme_transitions=transition_detail.merge(extreme_keys,on=["part","sid_src"],how="inner")
extreme_transitions["absolute_residual"]=extreme_transitions.exact_pair_centered_residual.abs()
extreme_transitions=(extreme_transitions.sort_values(
    ["part","sid_src","absolute_residual"],ascending=[True,True,False])
    .groupby(["part","sid_src"],as_index=False).head(2)
    .drop(columns="absolute_residual"))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("Most source-retaining and donor-receptive species/part cells, with before-to-after components:")
display(species_extremes.round(3))
print("Two strongest old-value to inserted-value transitions inside each printed extreme cell:")
display(extreme_transitions.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)

PARITY_RUN.append(dict(figure='f8',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_transition=(R.groupby(["part","sid_src"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
transition_detail=(R.groupby(["part","sid_src","var_src","var_donor"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
species_extremes=pd.concat([
    pd.concat([group.nsmallest(2,'exact_pair_centered_residual'),
               group.nlargest(2,'exact_pair_centered_residual')])
    for _,group in species_transition.groupby('part')
]).sort_values(['part','exact_pair_centered_residual'])
extreme_keys=species_extremes[["part","sid_src"]].drop_duplicates()
extreme_transitions=transition_detail.merge(extreme_keys,on=["part","sid_src"],how="inner")
extreme_transitions["absolute_residual"]=extreme_transitions.exact_pair_centered_residual.abs()
extreme_transitions=(extreme_transitions.sort_values(
    ["part","sid_src","absolute_residual"],ascending=[True,True,False])
    .groupby(["part","sid_src"],as_index=False).head(2)
    .drop(columns="absolute_residual"))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("Most source-retaining and donor-receptive species/part cells, with before-to-after components:")
display(species_extremes.round(3))
print("Two strongest old-value to inserted-value transitions inside each printed extreme cell:")
display(extreme_transitions.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)

PARITY_RUN.append(dict(figure='f8',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_transition=(R.groupby(["part","sid_src"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
transition_detail=(R.groupby(["part","sid_src","var_src","var_donor"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
species_extremes=pd.concat([
    pd.concat([group.nsmallest(2,'exact_pair_centered_residual'),
               group.nlargest(2,'exact_pair_centered_residual')])
    for _,group in species_transition.groupby('part')
]).sort_values(['part','exact_pair_centered_residual'])
extreme_keys=species_extremes[["part","sid_src"]].drop_duplicates()
extreme_transitions=transition_detail.merge(extreme_keys,on=["part","sid_src"],how="inner")
extreme_transitions["absolute_residual"]=extreme_transitions.exact_pair_centered_residual.abs()
extreme_transitions=(extreme_transitions.sort_values(
    ["part","sid_src","absolute_residual"],ascending=[True,True,False])
    .groupby(["part","sid_src"],as_index=False).head(2)
    .drop(columns="absolute_residual"))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("Most source-retaining and donor-receptive species/part cells, with before-to-after components:")
display(species_extremes.round(3))
print("Two strongest old-value to inserted-value transitions inside each printed extreme cell:")
display(extreme_transitions.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)

PARITY_RUN.append(dict(figure='f8',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_transition=(R.groupby(["part","sid_src"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
transition_detail=(R.groupby(["part","sid_src","var_src","var_donor"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
species_extremes=pd.concat([
    pd.concat([group.nsmallest(2,'exact_pair_centered_residual'),
               group.nlargest(2,'exact_pair_centered_residual')])
    for _,group in species_transition.groupby('part')
]).sort_values(['part','exact_pair_centered_residual'])
extreme_keys=species_extremes[["part","sid_src"]].drop_duplicates()
extreme_transitions=transition_detail.merge(extreme_keys,on=["part","sid_src"],how="inner")
extreme_transitions["absolute_residual"]=extreme_transitions.exact_pair_centered_residual.abs()
extreme_transitions=(extreme_transitions.sort_values(
    ["part","sid_src","absolute_residual"],ascending=[True,True,False])
    .groupby(["part","sid_src"],as_index=False).head(2)
    .drop(columns="absolute_residual"))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("Most source-retaining and donor-receptive species/part cells, with before-to-after components:")
display(species_extremes.round(3))
print("Two strongest old-value to inserted-value transitions inside each printed extreme cell:")
display(extreme_transitions.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)

PARITY_RUN.append(dict(figure='f8',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_transition=(R.groupby(["part","sid_src"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
transition_detail=(R.groupby(["part","sid_src","var_src","var_donor"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
species_extremes=pd.concat([
    pd.concat([group.nsmallest(2,'exact_pair_centered_residual'),
               group.nlargest(2,'exact_pair_centered_residual')])
    for _,group in species_transition.groupby('part')
]).sort_values(['part','exact_pair_centered_residual'])
extreme_keys=species_extremes[["part","sid_src"]].drop_duplicates()
extreme_transitions=transition_detail.merge(extreme_keys,on=["part","sid_src"],how="inner")
extreme_transitions["absolute_residual"]=extreme_transitions.exact_pair_centered_residual.abs()
extreme_transitions=(extreme_transitions.sort_values(
    ["part","sid_src","absolute_residual"],ascending=[True,True,False])
    .groupby(["part","sid_src"],as_index=False).head(2)
    .drop(columns="absolute_residual"))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("Most source-retaining and donor-receptive species/part cells, with before-to-after components:")
display(species_extremes.round(3))
print("Two strongest old-value to inserted-value transitions inside each printed extreme cell:")
display(extreme_transitions.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)

PARITY_RUN.append(dict(figure='f8',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_transition=(R.groupby(["part","sid_src"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
transition_detail=(R.groupby(["part","sid_src","var_src","var_donor"]).agg(
    n=("margin","size"),mean_starting_margin=("m_orig","mean"),
    mean_donorward_movement=("response_delta","mean"),
    mean_final_margin=("margin","mean"),mean_visible_pixels=("pixel_count_cf","mean"),
    exact_pair_centered_residual=("margin_after_value_pair","mean")).reset_index())
species_extremes=pd.concat([
    pd.concat([group.nsmallest(2,'exact_pair_centered_residual'),
               group.nlargest(2,'exact_pair_centered_residual')])
    for _,group in species_transition.groupby('part')
]).sort_values(['part','exact_pair_centered_residual'])
extreme_keys=species_extremes[["part","sid_src"]].drop_duplicates()
extreme_transitions=transition_detail.merge(extreme_keys,on=["part","sid_src"],how="inner")
extreme_transitions["absolute_residual"]=extreme_transitions.exact_pair_centered_residual.abs()
extreme_transitions=(extreme_transitions.sort_values(
    ["part","sid_src","absolute_residual"],ascending=[True,True,False])
    .groupby(["part","sid_src"],as_index=False).head(2)
    .drop(columns="absolute_residual"))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("Most source-retaining and donor-receptive species/part cells, with before-to-after components:")
display(species_extremes.round(3))
print("Two strongest old-value to inserted-value transitions inside each printed extreme cell:")
display(extreme_transitions.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)

PARITY_RUN.append(dict(figure='f8',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f8b

## 8b · How much species identity is recoverable from the learned concept vector?

**Question.** How much species identity is recoverable from the learned concept vector?

**Variables and prediction.** After the CBM is finished, train three read-only diagnostic classifiers. Repeat the test once with the complete five-part recipe and once with each part alone. Grey receives official yes/no answers c; solid color receives model raw scores z; outline receives each raw score after subtracting the training-fold average for the same yes/no answer. Raw or residual accuracy above the known-label control means score magnitudes reveal species beyond the nominal concept pattern. It does not say which pixels produced them, whether the saved class head uses them, or whether they caused a swap failure.

**Method.** Use one fixed stratified 70/30 split of the held-out prediction population.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 8b · How much species identity is recoverable from the learned concept vector?

**How to read the figure.** The y-axis is held-out species-classification accuracy. For every block, grey
uses processed 0/1 concept labels, color uses learned raw logits, and the
outlined bar uses raw logits after the training-fold mean for the same 0/1
label has been removed. The grey bar is the structural control: with balanced
FunnyBird species and `K` mutually exclusive values for one part, it is
approximately `K/50` (tail has 9 values, so 9/50=0.18), not 1/50. The residual
bar asks whether score magnitudes still identify species after the nominal
label bucket is removed. This diagnoses available information only; it does
not prove that the saved CBM uses it or that it caused backwash.


### Before Figure 8b: what exactly are the grey and colored bars?

Each image has 26 official yes/no answers, written `c`. The word
**binary** means only that an answer is 0 for “absent” or 1 for “present.”
For example, a bird can have `beak_0=1` and the other beak values equal
to 0. Across the five parts, these 26 answers are simply a long way to
record the bird's complete tail + wing + beak + foot + eye recipe. They
are dataset facts, not model scores.

**Why can all official answers identify 100% of species while one part
cannot?** The 50 synthetic species have distinct *combinations* of the
five part values. Several species can share the same tail, so tail alone
leaves several possible species. Adding wing, beak, foot, and eye can
make the complete combination unique. It is like a five-digit code:
one digit is shared by many records, while all five digits together can
identify one record. Therefore the 100% bar for all five parts is
expected dataset structure; it does not mean any single concept head is
making a 50-species prediction.

We train separate diagnostic classifiers whose target is the species `y`:

- **grey bar:** input is the corresponding official yes/no answers `c`;
- **solid colored bar:** input is the corresponding learned raw scores `z`;
- **outlined bar:** input is `z` after subtracting the training-fold mean
  for the same exact concept and 0/1 label. This asks whether magnitudes
  still identify species *within* the official label buckets;
- **bar height:** held-out species accuracy of that diagnostic classifier.

Thus “species information is present” means only that a classifier can guess
species from the supplied numbers better than chance. It does **not** mean the
saved CBM classified the image with that accuracy, and it does **not** measure
whether a concept used its named pixels.

Example: suppose a purple tail is shared by species 4, 12, 19, 31, and
44. The official tail answer can narrow the choice to those five species
but cannot say which of the five is present. If the nine raw tail scores
nevertheless differ systematically among those species, a diagnostic can
do better than the official tail answers. That extra performance is the
within-bucket species information being tested.

**One three-bar numerical example.** Imagine all five species in that
purple-tail group have the same official tail answer:

- grey sees only “purple tail=yes,” so it cannot distinguish the five;
- solid color sees the actual nine scores, for example
  `[-4,-3,+6,-2,...]`, and may recognize a species-specific score pattern;
- outline first subtracts the average nine-score pattern of all
  purple-tail training birds. If the remaining deviations still identify
  species, the score magnitudes contain information beyond “purple=yes.”

The bars therefore mean **nominal answer**, **all learned numerical
detail**, and **numerical detail left after the nominal answer is removed**.

A single part block is supplied as several numbers to a multinomial
logistic regression. For example, the nine tail scores become nine input
columns, and the diagnostic fits 50 weighted sums—one per species. We are
not zeroing other weights in the saved CBM because this is a new diagnostic
classifier. Figure 8c instead asks the swap-specific question that this
ordinary-image probe cannot answer: whether post-swap scores retain the
unchanged source species after exact source and donor values are controlled.

The held-out probe population is only 30% of 500 images: 150 images, or
roughly three per species. Therefore small differences between part bars
can correspond to only a few images. Use this test to establish that
information is available, not to claim a precise causal ranking.

> **IMPORTANT: Species leakage makes backwash possible, but leakage alone does
> not cause it. Wing is the clearest counterexample: wing `z` reveals species,
> yet the controlled swaps show strong grounding.**

**Connection to the MCBM/new-loss hypothesis.** A minimality loss should
reduce the outlined bar by making images with the same official concept
answer produce more similar internal representations. Notebook 03 tests
that prediction; it is not assumed here. A post-hoc simulation could
replace each raw score with its label-conditioned mean or an ideal
`-3/+3` code and pass that vector through the unchanged species head. That
would test downstream sensitivity to removing within-label detail, but it
would not prove that a trainable model looks at the correct pixels. A
stronger future loss would use the known swap directly: raise the inserted
value, lower the removed value, and keep unrelated coordinates stable.
That is a swap-consistency hypothesis, not an MCBM result in this chapter.

What predicts grounding is measured separately: `response_delta`, final margin
`m_cf`, target-part visibility, label/mask conflict, and exact donor-value
recognition. Figure 8b is an availability/control diagnostic, not that outcome.


- **Method in one line:** We fitted three new multinomial logistic-regression diagnostic classifiers per concept block on 70% of the frozen CBM's held-out images and tested them on the same remaining 30%: one used binary labels, one raw logits, and one within-label residual logits; the CBM itself was not retrained or altered.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
show_standard('f8b','f3a0afce515090eb5f62dbe53c8e34efa39fb2c88cec3e4b6a8ca87b3e6c9f04')


In [ ]:
# ALT: MCBM gamma=0: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among held-out ordinary images (N printed above) (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))

PARITY_CACHE[GAMMA]['PROBE']=PROBE
PARITY_RUN.append(dict(figure='f8b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among held-out ordinary images (N printed above) (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))

PARITY_CACHE[GAMMA]['PROBE']=PROBE
PARITY_RUN.append(dict(figure='f8b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among held-out ordinary images (N printed above) (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))

PARITY_CACHE[GAMMA]['PROBE']=PROBE
PARITY_RUN.append(dict(figure='f8b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among held-out ordinary images (N printed above) (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))

PARITY_CACHE[GAMMA]['PROBE']=PROBE
PARITY_RUN.append(dict(figure='f8b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among held-out ordinary images (N printed above) (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))

PARITY_CACHE[GAMMA]['PROBE']=PROBE
PARITY_RUN.append(dict(figure='f8b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among held-out ordinary images (N printed above) (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))

PARITY_CACHE[GAMMA]['PROBE']=PROBE
PARITY_RUN.append(dict(figure='f8b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction r8b-compare

Compare the already defined decoding accuracy, mean donorward movement, exact inserted-value recognition, and strict controlled-backwash fraction for each part. No additional model is fitted.

**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
show_standard('r8b-compare','bea7fc15c6438349ccb31a3645ab97bcc96e49cd702fa50177a08b262ac588d7')


In [ ]:
# ALT: MCBM gamma=0: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
PROBE=PARITY_CACHE[GAMMA]['PROBE']
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(GROUNDING_COMPARISON.round(4))

PARITY_RUN.append(dict(figure='r8b-compare',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
PROBE=PARITY_CACHE[GAMMA]['PROBE']
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(GROUNDING_COMPARISON.round(4))

PARITY_RUN.append(dict(figure='r8b-compare',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
PROBE=PARITY_CACHE[GAMMA]['PROBE']
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(GROUNDING_COMPARISON.round(4))

PARITY_RUN.append(dict(figure='r8b-compare',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
PROBE=PARITY_CACHE[GAMMA]['PROBE']
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(GROUNDING_COMPARISON.round(4))

PARITY_RUN.append(dict(figure='r8b-compare',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
PROBE=PARITY_CACHE[GAMMA]['PROBE']
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(GROUNDING_COMPARISON.round(4))

PARITY_RUN.append(dict(figure='r8b-compare',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
PROBE=PARITY_CACHE[GAMMA]['PROBE']
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(GROUNDING_COMPARISON.round(4))

PARITY_RUN.append(dict(figure='r8b-compare',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f8c-source


**Question/prediction:** does compression reduce extra species information and saved-head sensitivity together?
Panel A/B fit **new** fivefold species logistic regressions. For each coordinate,
r_ij=z_ij−mean_train(z_j|c_j=c_ij). Give one probe c, the other [c,r].
Log-loss L=−mean_i log P_probe(y_i); gain=L(c)−L(c,r), in natural-log units.
Positive gain means improved held-out prediction, not a percentage of species information.
A uses all coordinates in that part; B uses exactly three (all subsets up to 40,
otherwise 40 deterministic sampled subsets); bars average gains, lines show subset
min/max, **not confidence intervals**. The h versions are printed separately.

C/D reuse the **unchanged original MCBM species MLP F(h)**, not a probe and not Wz+b.
Replace a part's h_j by training-fold mean(h_j|c_j). Example: positive tail_4 values
[2,3,4] give mean3; replace a held-out4.2 with3. True labels determine the bucket;
this is not guaranteed to preserve the model's predicted sign or be deployable without labels.
C compares correct-species fractions before and after replacing all26. D shows
mean_i[0.5 sum_species |P_before−P_after|], replacing each block separately.
For [0.8,0.2]→[0.6,0.4], mass moved=.2. This is replacement sensitivity, not pure
species-information usage. Colors preserve part identity; black is all26.
Denominator: all ordinary exported images, split five ways stratified by species,
seed20260903; all centering/probe fitting uses only the other folds. N/mean/SD
for absent/present h AND z are printed for every coordinate before the plots.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
show_standard('f8c-source','15ff6768c379631ea81efc98f8297b0884721bf5f226e16593d81ebda6fccc2b')


In [ ]:
# ALT: MCBM gamma=0: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

# A/B: same conditional-information diagnostic on raw concept z as Standard.
FULL_WIDTH_INFORMATION,EQUAL_WIDTH_INFORMATION=conditional_information(z_saved,c_saved,y_saved,SPANS,progress=lambda s:print('z:',s,flush=True))
# Additional h table: the representation the MCBM species head actually consumes.
H_INFORMATION,H_EQUAL_WIDTH=conditional_information(h_saved,c_saved,y_saved,SPANS,progress=lambda s:print('h:',s,flush=True))
HEAD_USE=replacement_use(h_saved,c_saved,y_saved,ordinary['y_probability'],saved_head,SPANS)
display(score_reference(h_saved,z_saved,c_saved,CONCEPT_NAMES).round(4))
display(FULL_WIDTH_INFORMATION.round(4)); display(EQUAL_WIDTH_INFORMATION.round(4))
print('Additional internal-slot h information (not substituted for the raw-z panels):')
display(H_INFORMATION.round(4)); display(H_EQUAL_WIDTH.round(4)); display(HEAD_USE.round(4))
full=FULL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
equal=EQUAL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
head=HEAD_USE.set_index('replaced_block')
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(ORDER,full.conditional_logloss_gain,color=[COLORS[p] for p in ORDER])
axes[0,0].set_ylabel('held-out log-loss improvement'); axes[0,0].set_title('A · Extra species information after 0/1 labels are known')
mean=equal.mean_conditional_gain.to_numpy()
axes[0,1].bar(ORDER,mean,color=[COLORS[p] for p in ORDER])
axes[0,1].errorbar(np.arange(5),mean,
 yerr=np.stack([mean-equal.min_conditional_gain.to_numpy(),equal.max_conditional_gain.to_numpy()-mean]),fmt='none',color='black')
axes[0,1].set_ylabel('mean gain; line = subset range'); axes[0,1].set_title('B · Same three-coordinate budget for every part')
axes[1,0].bar(['raw internal h','label-conditioned\nmeans'],
 [head.loc['all 26','raw_accuracy'],head.loc['all 26','accuracy_after_replacement']],color=['#333333','#BBBBBB'])
axes[1,0].set_ylim(0,1); axes[1,0].set_ylabel('saved MCBM head accuracy')
axes[1,0].set_title('C · What the unchanged nonlinear species head uses')
blocks=['all 26']+ORDER
axes[1,1].bar(blocks,head.loc[blocks,'mean_probability_mass_moved'],color=['#333333']+[COLORS[p] for p in ORDER])
axes[1,1].set_ylabel('mean probability mass moved'); axes[1,1].set_title('D · Sensitivity to replacing within-label magnitudes')
for ax in axes.flat: ax.axhline(0,color='black',lw=.5)
fig.suptitle(f'MCBM gamma={GAMMA:g} · Information available versus saved-head sensitivity')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['HEAD_USE']=HEAD_USE
PARITY_CACHE[GAMMA]['FULL_WIDTH_INFORMATION']=FULL_WIDTH_INFORMATION
PARITY_CACHE[GAMMA]['EQUAL_WIDTH_INFORMATION']=EQUAL_WIDTH_INFORMATION
PARITY_RUN.append(dict(figure='f8c-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

# A/B: same conditional-information diagnostic on raw concept z as Standard.
FULL_WIDTH_INFORMATION,EQUAL_WIDTH_INFORMATION=conditional_information(z_saved,c_saved,y_saved,SPANS,progress=lambda s:print('z:',s,flush=True))
# Additional h table: the representation the MCBM species head actually consumes.
H_INFORMATION,H_EQUAL_WIDTH=conditional_information(h_saved,c_saved,y_saved,SPANS,progress=lambda s:print('h:',s,flush=True))
HEAD_USE=replacement_use(h_saved,c_saved,y_saved,ordinary['y_probability'],saved_head,SPANS)
display(score_reference(h_saved,z_saved,c_saved,CONCEPT_NAMES).round(4))
display(FULL_WIDTH_INFORMATION.round(4)); display(EQUAL_WIDTH_INFORMATION.round(4))
print('Additional internal-slot h information (not substituted for the raw-z panels):')
display(H_INFORMATION.round(4)); display(H_EQUAL_WIDTH.round(4)); display(HEAD_USE.round(4))
full=FULL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
equal=EQUAL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
head=HEAD_USE.set_index('replaced_block')
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(ORDER,full.conditional_logloss_gain,color=[COLORS[p] for p in ORDER])
axes[0,0].set_ylabel('held-out log-loss improvement'); axes[0,0].set_title('A · Extra species information after 0/1 labels are known')
mean=equal.mean_conditional_gain.to_numpy()
axes[0,1].bar(ORDER,mean,color=[COLORS[p] for p in ORDER])
axes[0,1].errorbar(np.arange(5),mean,
 yerr=np.stack([mean-equal.min_conditional_gain.to_numpy(),equal.max_conditional_gain.to_numpy()-mean]),fmt='none',color='black')
axes[0,1].set_ylabel('mean gain; line = subset range'); axes[0,1].set_title('B · Same three-coordinate budget for every part')
axes[1,0].bar(['raw internal h','label-conditioned\nmeans'],
 [head.loc['all 26','raw_accuracy'],head.loc['all 26','accuracy_after_replacement']],color=['#333333','#BBBBBB'])
axes[1,0].set_ylim(0,1); axes[1,0].set_ylabel('saved MCBM head accuracy')
axes[1,0].set_title('C · What the unchanged nonlinear species head uses')
blocks=['all 26']+ORDER
axes[1,1].bar(blocks,head.loc[blocks,'mean_probability_mass_moved'],color=['#333333']+[COLORS[p] for p in ORDER])
axes[1,1].set_ylabel('mean probability mass moved'); axes[1,1].set_title('D · Sensitivity to replacing within-label magnitudes')
for ax in axes.flat: ax.axhline(0,color='black',lw=.5)
fig.suptitle(f'MCBM gamma={GAMMA:g} · Information available versus saved-head sensitivity')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['HEAD_USE']=HEAD_USE
PARITY_CACHE[GAMMA]['FULL_WIDTH_INFORMATION']=FULL_WIDTH_INFORMATION
PARITY_CACHE[GAMMA]['EQUAL_WIDTH_INFORMATION']=EQUAL_WIDTH_INFORMATION
PARITY_RUN.append(dict(figure='f8c-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

# A/B: same conditional-information diagnostic on raw concept z as Standard.
FULL_WIDTH_INFORMATION,EQUAL_WIDTH_INFORMATION=conditional_information(z_saved,c_saved,y_saved,SPANS,progress=lambda s:print('z:',s,flush=True))
# Additional h table: the representation the MCBM species head actually consumes.
H_INFORMATION,H_EQUAL_WIDTH=conditional_information(h_saved,c_saved,y_saved,SPANS,progress=lambda s:print('h:',s,flush=True))
HEAD_USE=replacement_use(h_saved,c_saved,y_saved,ordinary['y_probability'],saved_head,SPANS)
display(score_reference(h_saved,z_saved,c_saved,CONCEPT_NAMES).round(4))
display(FULL_WIDTH_INFORMATION.round(4)); display(EQUAL_WIDTH_INFORMATION.round(4))
print('Additional internal-slot h information (not substituted for the raw-z panels):')
display(H_INFORMATION.round(4)); display(H_EQUAL_WIDTH.round(4)); display(HEAD_USE.round(4))
full=FULL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
equal=EQUAL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
head=HEAD_USE.set_index('replaced_block')
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(ORDER,full.conditional_logloss_gain,color=[COLORS[p] for p in ORDER])
axes[0,0].set_ylabel('held-out log-loss improvement'); axes[0,0].set_title('A · Extra species information after 0/1 labels are known')
mean=equal.mean_conditional_gain.to_numpy()
axes[0,1].bar(ORDER,mean,color=[COLORS[p] for p in ORDER])
axes[0,1].errorbar(np.arange(5),mean,
 yerr=np.stack([mean-equal.min_conditional_gain.to_numpy(),equal.max_conditional_gain.to_numpy()-mean]),fmt='none',color='black')
axes[0,1].set_ylabel('mean gain; line = subset range'); axes[0,1].set_title('B · Same three-coordinate budget for every part')
axes[1,0].bar(['raw internal h','label-conditioned\nmeans'],
 [head.loc['all 26','raw_accuracy'],head.loc['all 26','accuracy_after_replacement']],color=['#333333','#BBBBBB'])
axes[1,0].set_ylim(0,1); axes[1,0].set_ylabel('saved MCBM head accuracy')
axes[1,0].set_title('C · What the unchanged nonlinear species head uses')
blocks=['all 26']+ORDER
axes[1,1].bar(blocks,head.loc[blocks,'mean_probability_mass_moved'],color=['#333333']+[COLORS[p] for p in ORDER])
axes[1,1].set_ylabel('mean probability mass moved'); axes[1,1].set_title('D · Sensitivity to replacing within-label magnitudes')
for ax in axes.flat: ax.axhline(0,color='black',lw=.5)
fig.suptitle(f'MCBM gamma={GAMMA:g} · Information available versus saved-head sensitivity')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['HEAD_USE']=HEAD_USE
PARITY_CACHE[GAMMA]['FULL_WIDTH_INFORMATION']=FULL_WIDTH_INFORMATION
PARITY_CACHE[GAMMA]['EQUAL_WIDTH_INFORMATION']=EQUAL_WIDTH_INFORMATION
PARITY_RUN.append(dict(figure='f8c-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

# A/B: same conditional-information diagnostic on raw concept z as Standard.
FULL_WIDTH_INFORMATION,EQUAL_WIDTH_INFORMATION=conditional_information(z_saved,c_saved,y_saved,SPANS,progress=lambda s:print('z:',s,flush=True))
# Additional h table: the representation the MCBM species head actually consumes.
H_INFORMATION,H_EQUAL_WIDTH=conditional_information(h_saved,c_saved,y_saved,SPANS,progress=lambda s:print('h:',s,flush=True))
HEAD_USE=replacement_use(h_saved,c_saved,y_saved,ordinary['y_probability'],saved_head,SPANS)
display(score_reference(h_saved,z_saved,c_saved,CONCEPT_NAMES).round(4))
display(FULL_WIDTH_INFORMATION.round(4)); display(EQUAL_WIDTH_INFORMATION.round(4))
print('Additional internal-slot h information (not substituted for the raw-z panels):')
display(H_INFORMATION.round(4)); display(H_EQUAL_WIDTH.round(4)); display(HEAD_USE.round(4))
full=FULL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
equal=EQUAL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
head=HEAD_USE.set_index('replaced_block')
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(ORDER,full.conditional_logloss_gain,color=[COLORS[p] for p in ORDER])
axes[0,0].set_ylabel('held-out log-loss improvement'); axes[0,0].set_title('A · Extra species information after 0/1 labels are known')
mean=equal.mean_conditional_gain.to_numpy()
axes[0,1].bar(ORDER,mean,color=[COLORS[p] for p in ORDER])
axes[0,1].errorbar(np.arange(5),mean,
 yerr=np.stack([mean-equal.min_conditional_gain.to_numpy(),equal.max_conditional_gain.to_numpy()-mean]),fmt='none',color='black')
axes[0,1].set_ylabel('mean gain; line = subset range'); axes[0,1].set_title('B · Same three-coordinate budget for every part')
axes[1,0].bar(['raw internal h','label-conditioned\nmeans'],
 [head.loc['all 26','raw_accuracy'],head.loc['all 26','accuracy_after_replacement']],color=['#333333','#BBBBBB'])
axes[1,0].set_ylim(0,1); axes[1,0].set_ylabel('saved MCBM head accuracy')
axes[1,0].set_title('C · What the unchanged nonlinear species head uses')
blocks=['all 26']+ORDER
axes[1,1].bar(blocks,head.loc[blocks,'mean_probability_mass_moved'],color=['#333333']+[COLORS[p] for p in ORDER])
axes[1,1].set_ylabel('mean probability mass moved'); axes[1,1].set_title('D · Sensitivity to replacing within-label magnitudes')
for ax in axes.flat: ax.axhline(0,color='black',lw=.5)
fig.suptitle(f'MCBM gamma={GAMMA:g} · Information available versus saved-head sensitivity')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['HEAD_USE']=HEAD_USE
PARITY_CACHE[GAMMA]['FULL_WIDTH_INFORMATION']=FULL_WIDTH_INFORMATION
PARITY_CACHE[GAMMA]['EQUAL_WIDTH_INFORMATION']=EQUAL_WIDTH_INFORMATION
PARITY_RUN.append(dict(figure='f8c-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

# A/B: same conditional-information diagnostic on raw concept z as Standard.
FULL_WIDTH_INFORMATION,EQUAL_WIDTH_INFORMATION=conditional_information(z_saved,c_saved,y_saved,SPANS,progress=lambda s:print('z:',s,flush=True))
# Additional h table: the representation the MCBM species head actually consumes.
H_INFORMATION,H_EQUAL_WIDTH=conditional_information(h_saved,c_saved,y_saved,SPANS,progress=lambda s:print('h:',s,flush=True))
HEAD_USE=replacement_use(h_saved,c_saved,y_saved,ordinary['y_probability'],saved_head,SPANS)
display(score_reference(h_saved,z_saved,c_saved,CONCEPT_NAMES).round(4))
display(FULL_WIDTH_INFORMATION.round(4)); display(EQUAL_WIDTH_INFORMATION.round(4))
print('Additional internal-slot h information (not substituted for the raw-z panels):')
display(H_INFORMATION.round(4)); display(H_EQUAL_WIDTH.round(4)); display(HEAD_USE.round(4))
full=FULL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
equal=EQUAL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
head=HEAD_USE.set_index('replaced_block')
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(ORDER,full.conditional_logloss_gain,color=[COLORS[p] for p in ORDER])
axes[0,0].set_ylabel('held-out log-loss improvement'); axes[0,0].set_title('A · Extra species information after 0/1 labels are known')
mean=equal.mean_conditional_gain.to_numpy()
axes[0,1].bar(ORDER,mean,color=[COLORS[p] for p in ORDER])
axes[0,1].errorbar(np.arange(5),mean,
 yerr=np.stack([mean-equal.min_conditional_gain.to_numpy(),equal.max_conditional_gain.to_numpy()-mean]),fmt='none',color='black')
axes[0,1].set_ylabel('mean gain; line = subset range'); axes[0,1].set_title('B · Same three-coordinate budget for every part')
axes[1,0].bar(['raw internal h','label-conditioned\nmeans'],
 [head.loc['all 26','raw_accuracy'],head.loc['all 26','accuracy_after_replacement']],color=['#333333','#BBBBBB'])
axes[1,0].set_ylim(0,1); axes[1,0].set_ylabel('saved MCBM head accuracy')
axes[1,0].set_title('C · What the unchanged nonlinear species head uses')
blocks=['all 26']+ORDER
axes[1,1].bar(blocks,head.loc[blocks,'mean_probability_mass_moved'],color=['#333333']+[COLORS[p] for p in ORDER])
axes[1,1].set_ylabel('mean probability mass moved'); axes[1,1].set_title('D · Sensitivity to replacing within-label magnitudes')
for ax in axes.flat: ax.axhline(0,color='black',lw=.5)
fig.suptitle(f'MCBM gamma={GAMMA:g} · Information available versus saved-head sensitivity')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['HEAD_USE']=HEAD_USE
PARITY_CACHE[GAMMA]['FULL_WIDTH_INFORMATION']=FULL_WIDTH_INFORMATION
PARITY_CACHE[GAMMA]['EQUAL_WIDTH_INFORMATION']=EQUAL_WIDTH_INFORMATION
PARITY_RUN.append(dict(figure='f8c-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Four panels separating species information available in full and equal-width part blocks from accuracy and probability movement in the unchanged saved CBM species head after within-label magnitudes are removed.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

# A/B: same conditional-information diagnostic on raw concept z as Standard.
FULL_WIDTH_INFORMATION,EQUAL_WIDTH_INFORMATION=conditional_information(z_saved,c_saved,y_saved,SPANS,progress=lambda s:print('z:',s,flush=True))
# Additional h table: the representation the MCBM species head actually consumes.
H_INFORMATION,H_EQUAL_WIDTH=conditional_information(h_saved,c_saved,y_saved,SPANS,progress=lambda s:print('h:',s,flush=True))
HEAD_USE=replacement_use(h_saved,c_saved,y_saved,ordinary['y_probability'],saved_head,SPANS)
display(score_reference(h_saved,z_saved,c_saved,CONCEPT_NAMES).round(4))
display(FULL_WIDTH_INFORMATION.round(4)); display(EQUAL_WIDTH_INFORMATION.round(4))
print('Additional internal-slot h information (not substituted for the raw-z panels):')
display(H_INFORMATION.round(4)); display(H_EQUAL_WIDTH.round(4)); display(HEAD_USE.round(4))
full=FULL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
equal=EQUAL_WIDTH_INFORMATION.set_index('part').reindex(ORDER)
head=HEAD_USE.set_index('replaced_block')
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(ORDER,full.conditional_logloss_gain,color=[COLORS[p] for p in ORDER])
axes[0,0].set_ylabel('held-out log-loss improvement'); axes[0,0].set_title('A · Extra species information after 0/1 labels are known')
mean=equal.mean_conditional_gain.to_numpy()
axes[0,1].bar(ORDER,mean,color=[COLORS[p] for p in ORDER])
axes[0,1].errorbar(np.arange(5),mean,
 yerr=np.stack([mean-equal.min_conditional_gain.to_numpy(),equal.max_conditional_gain.to_numpy()-mean]),fmt='none',color='black')
axes[0,1].set_ylabel('mean gain; line = subset range'); axes[0,1].set_title('B · Same three-coordinate budget for every part')
axes[1,0].bar(['raw internal h','label-conditioned\nmeans'],
 [head.loc['all 26','raw_accuracy'],head.loc['all 26','accuracy_after_replacement']],color=['#333333','#BBBBBB'])
axes[1,0].set_ylim(0,1); axes[1,0].set_ylabel('saved MCBM head accuracy')
axes[1,0].set_title('C · What the unchanged nonlinear species head uses')
blocks=['all 26']+ORDER
axes[1,1].bar(blocks,head.loc[blocks,'mean_probability_mass_moved'],color=['#333333']+[COLORS[p] for p in ORDER])
axes[1,1].set_ylabel('mean probability mass moved'); axes[1,1].set_title('D · Sensitivity to replacing within-label magnitudes')
for ax in axes.flat: ax.axhline(0,color='black',lw=.5)
fig.suptitle(f'MCBM gamma={GAMMA:g} · Information available versus saved-head sensitivity')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['HEAD_USE']=HEAD_USE
PARITY_CACHE[GAMMA]['FULL_WIDTH_INFORMATION']=FULL_WIDTH_INFORMATION
PARITY_CACHE[GAMMA]['EQUAL_WIDTH_INFORMATION']=EQUAL_WIDTH_INFORMATION
PARITY_RUN.append(dict(figure='f8c-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f8d-source


**Question/prediction:** after swapping pixels, do other same-part slots still favor
the old species through the saved classifier? Compare distributions **between parts**;
within-part correlations are retained later in the appendix, not the main test.

For tail_2→tail_7 there are always nine fixed tail outputs. Keep slots2 and7 unchanged.
For this diagnostic only, replace the other seven internal h slots by their ordinary
mean when the corresponding c_j=0. Example: absent tail_4 averages−2.8; replace its
counterfactual value−1.2 by−2.8. No image or trained weight is changed.

Let G(h)=F_source(h)−F_donor(h), using the saved **nonlinear** MCBM species head.
e=G(h_cf)−G(h_erased). Positive e means the removed variation favored the source
in this replacement context. It is a joint finite intervention, **not** the sum of
fixed W_j contributions and not an additive decomposition of nonlinear interactions.
No new classifier is trained. The raw old/new concept margin m_cf is unchanged:
each q_j reads only its own untouched h_j. The species head cannot cause an upstream
concept error during this forward pass; task gradients during training are a different question.

A shows e in species-logit units; B shows 100[sigmoid(G_before)−sigmoid(G_after)]
in percentage points; C shows G_before. Boxes show quartiles, middle line median,
whiskers5–95%; omitted outliers are still in all summaries. Fractions positive,
means/medians, original counts, top-one changes and source→donor pair flips are printed.
Example: G_before=5,G_after=4 gives e=1 but both still choose the source pairwise.
Neither pairwise share nor an unrelated top-one change is a whole donor-species takeover.
All5000 swaps are included,1000 perpart; repeated originals are not independent seeds.
Part blocks contain different numbers of erased slots and head scales can differ;
absolute cross-gamma e is not a calibrated information measure. Erasure may remove
visibility or other useful variation, not only a pure species fingerprint.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
show_standard('f8d-source','d2abd857ab8be3370cbdf2c892fac7e4e915c4b9c9ccb04cc0b005c1d8b03c28')


In [ ]:
# ALT: MCBM gamma=0: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

h_cf=replay_counterfactual_h(S,GAMMA,CURATED,REPO)
EVIDENCE_ROWS=off_target_erasure(S,h_cf,h_saved,c_saved,saved_head,SPANS)
EVIDENCE_SUMMARY=EVIDENCE_ROWS.groupby('part').agg(
 n=('m_cf','size'), originals=('original_image','nunique'),
 mean_e=('off_target_source_evidence','mean'), median_e=('off_target_source_evidence','median'),
 fraction_e_positive=('off_target_source_evidence',lambda a:float((a>0).mean())),
 mean_pairwise_share_reduction=('pairwise_source_share_reduction','mean'),
 top1_change_rate=('top1_changed','mean'), source_to_donor_pair_flip_rate=('source_to_donor_pair_flip','mean')
).reindex(ORDER)
display(EVIDENCE_SUMMARY.round(4))
fig,axes=plt.subplots(1,3,figsize=(17,5))
for ax,column,scale,title in [
 (axes[0],'off_target_source_evidence',1,'A · Source-over-donor gap reduction after erasure'),
 (axes[1],'pairwise_source_share_reduction',100,'B · Pairwise source-share reduction (percentage points)'),
 (axes[2],'source_minus_donor_logit_before',1,'C · Source-over-donor species gap before erasure')]:
    vals=[EVIDENCE_ROWS.loc[EVIDENCE_ROWS.part==p,column].to_numpy()*scale for p in ORDER]
    boxes=ax.boxplot(vals,tick_labels=ORDER,patch_artist=True,showfliers=False,whis=(5,95))
    for patch,p in zip(boxes['boxes'],ORDER): patch.set_facecolor(COLORS[p]); patch.set_alpha(.65)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title,fontsize=9)
    ax.set_ylabel('percentage points' if scale==100 else 'species-logit units')
fig.suptitle(f'MCBM gamma={GAMMA:g} · Frozen-head off-target erasure, old/new slots untouched')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']=EVIDENCE_ROWS
PARITY_RUN.append(dict(figure='f8d-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

h_cf=replay_counterfactual_h(S,GAMMA,CURATED,REPO)
EVIDENCE_ROWS=off_target_erasure(S,h_cf,h_saved,c_saved,saved_head,SPANS)
EVIDENCE_SUMMARY=EVIDENCE_ROWS.groupby('part').agg(
 n=('m_cf','size'), originals=('original_image','nunique'),
 mean_e=('off_target_source_evidence','mean'), median_e=('off_target_source_evidence','median'),
 fraction_e_positive=('off_target_source_evidence',lambda a:float((a>0).mean())),
 mean_pairwise_share_reduction=('pairwise_source_share_reduction','mean'),
 top1_change_rate=('top1_changed','mean'), source_to_donor_pair_flip_rate=('source_to_donor_pair_flip','mean')
).reindex(ORDER)
display(EVIDENCE_SUMMARY.round(4))
fig,axes=plt.subplots(1,3,figsize=(17,5))
for ax,column,scale,title in [
 (axes[0],'off_target_source_evidence',1,'A · Source-over-donor gap reduction after erasure'),
 (axes[1],'pairwise_source_share_reduction',100,'B · Pairwise source-share reduction (percentage points)'),
 (axes[2],'source_minus_donor_logit_before',1,'C · Source-over-donor species gap before erasure')]:
    vals=[EVIDENCE_ROWS.loc[EVIDENCE_ROWS.part==p,column].to_numpy()*scale for p in ORDER]
    boxes=ax.boxplot(vals,tick_labels=ORDER,patch_artist=True,showfliers=False,whis=(5,95))
    for patch,p in zip(boxes['boxes'],ORDER): patch.set_facecolor(COLORS[p]); patch.set_alpha(.65)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title,fontsize=9)
    ax.set_ylabel('percentage points' if scale==100 else 'species-logit units')
fig.suptitle(f'MCBM gamma={GAMMA:g} · Frozen-head off-target erasure, old/new slots untouched')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']=EVIDENCE_ROWS
PARITY_RUN.append(dict(figure='f8d-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

h_cf=replay_counterfactual_h(S,GAMMA,CURATED,REPO)
EVIDENCE_ROWS=off_target_erasure(S,h_cf,h_saved,c_saved,saved_head,SPANS)
EVIDENCE_SUMMARY=EVIDENCE_ROWS.groupby('part').agg(
 n=('m_cf','size'), originals=('original_image','nunique'),
 mean_e=('off_target_source_evidence','mean'), median_e=('off_target_source_evidence','median'),
 fraction_e_positive=('off_target_source_evidence',lambda a:float((a>0).mean())),
 mean_pairwise_share_reduction=('pairwise_source_share_reduction','mean'),
 top1_change_rate=('top1_changed','mean'), source_to_donor_pair_flip_rate=('source_to_donor_pair_flip','mean')
).reindex(ORDER)
display(EVIDENCE_SUMMARY.round(4))
fig,axes=plt.subplots(1,3,figsize=(17,5))
for ax,column,scale,title in [
 (axes[0],'off_target_source_evidence',1,'A · Source-over-donor gap reduction after erasure'),
 (axes[1],'pairwise_source_share_reduction',100,'B · Pairwise source-share reduction (percentage points)'),
 (axes[2],'source_minus_donor_logit_before',1,'C · Source-over-donor species gap before erasure')]:
    vals=[EVIDENCE_ROWS.loc[EVIDENCE_ROWS.part==p,column].to_numpy()*scale for p in ORDER]
    boxes=ax.boxplot(vals,tick_labels=ORDER,patch_artist=True,showfliers=False,whis=(5,95))
    for patch,p in zip(boxes['boxes'],ORDER): patch.set_facecolor(COLORS[p]); patch.set_alpha(.65)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title,fontsize=9)
    ax.set_ylabel('percentage points' if scale==100 else 'species-logit units')
fig.suptitle(f'MCBM gamma={GAMMA:g} · Frozen-head off-target erasure, old/new slots untouched')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']=EVIDENCE_ROWS
PARITY_RUN.append(dict(figure='f8d-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

h_cf=replay_counterfactual_h(S,GAMMA,CURATED,REPO)
EVIDENCE_ROWS=off_target_erasure(S,h_cf,h_saved,c_saved,saved_head,SPANS)
EVIDENCE_SUMMARY=EVIDENCE_ROWS.groupby('part').agg(
 n=('m_cf','size'), originals=('original_image','nunique'),
 mean_e=('off_target_source_evidence','mean'), median_e=('off_target_source_evidence','median'),
 fraction_e_positive=('off_target_source_evidence',lambda a:float((a>0).mean())),
 mean_pairwise_share_reduction=('pairwise_source_share_reduction','mean'),
 top1_change_rate=('top1_changed','mean'), source_to_donor_pair_flip_rate=('source_to_donor_pair_flip','mean')
).reindex(ORDER)
display(EVIDENCE_SUMMARY.round(4))
fig,axes=plt.subplots(1,3,figsize=(17,5))
for ax,column,scale,title in [
 (axes[0],'off_target_source_evidence',1,'A · Source-over-donor gap reduction after erasure'),
 (axes[1],'pairwise_source_share_reduction',100,'B · Pairwise source-share reduction (percentage points)'),
 (axes[2],'source_minus_donor_logit_before',1,'C · Source-over-donor species gap before erasure')]:
    vals=[EVIDENCE_ROWS.loc[EVIDENCE_ROWS.part==p,column].to_numpy()*scale for p in ORDER]
    boxes=ax.boxplot(vals,tick_labels=ORDER,patch_artist=True,showfliers=False,whis=(5,95))
    for patch,p in zip(boxes['boxes'],ORDER): patch.set_facecolor(COLORS[p]); patch.set_alpha(.65)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title,fontsize=9)
    ax.set_ylabel('percentage points' if scale==100 else 'species-logit units')
fig.suptitle(f'MCBM gamma={GAMMA:g} · Frozen-head off-target erasure, old/new slots untouched')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']=EVIDENCE_ROWS
PARITY_RUN.append(dict(figure='f8d-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

h_cf=replay_counterfactual_h(S,GAMMA,CURATED,REPO)
EVIDENCE_ROWS=off_target_erasure(S,h_cf,h_saved,c_saved,saved_head,SPANS)
EVIDENCE_SUMMARY=EVIDENCE_ROWS.groupby('part').agg(
 n=('m_cf','size'), originals=('original_image','nunique'),
 mean_e=('off_target_source_evidence','mean'), median_e=('off_target_source_evidence','median'),
 fraction_e_positive=('off_target_source_evidence',lambda a:float((a>0).mean())),
 mean_pairwise_share_reduction=('pairwise_source_share_reduction','mean'),
 top1_change_rate=('top1_changed','mean'), source_to_donor_pair_flip_rate=('source_to_donor_pair_flip','mean')
).reindex(ORDER)
display(EVIDENCE_SUMMARY.round(4))
fig,axes=plt.subplots(1,3,figsize=(17,5))
for ax,column,scale,title in [
 (axes[0],'off_target_source_evidence',1,'A · Source-over-donor gap reduction after erasure'),
 (axes[1],'pairwise_source_share_reduction',100,'B · Pairwise source-share reduction (percentage points)'),
 (axes[2],'source_minus_donor_logit_before',1,'C · Source-over-donor species gap before erasure')]:
    vals=[EVIDENCE_ROWS.loc[EVIDENCE_ROWS.part==p,column].to_numpy()*scale for p in ORDER]
    boxes=ax.boxplot(vals,tick_labels=ORDER,patch_artist=True,showfliers=False,whis=(5,95))
    for patch,p in zip(boxes['boxes'],ORDER): patch.set_facecolor(COLORS[p]); patch.set_alpha(.65)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title,fontsize=9)
    ax.set_ylabel('percentage points' if scale==100 else 'species-logit units')
fig.suptitle(f'MCBM gamma={GAMMA:g} · Frozen-head off-target erasure, old/new slots untouched')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']=EVIDENCE_ROWS
PARITY_RUN.append(dict(figure='f8d-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Three box-plot panels comparing off-target source-over-donor class evidence, its direct pairwise probability consequence after erasure, and the pre-erasure source-versus-donor class-logit gap that sets the probability scale.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)

h_cf=replay_counterfactual_h(S,GAMMA,CURATED,REPO)
EVIDENCE_ROWS=off_target_erasure(S,h_cf,h_saved,c_saved,saved_head,SPANS)
EVIDENCE_SUMMARY=EVIDENCE_ROWS.groupby('part').agg(
 n=('m_cf','size'), originals=('original_image','nunique'),
 mean_e=('off_target_source_evidence','mean'), median_e=('off_target_source_evidence','median'),
 fraction_e_positive=('off_target_source_evidence',lambda a:float((a>0).mean())),
 mean_pairwise_share_reduction=('pairwise_source_share_reduction','mean'),
 top1_change_rate=('top1_changed','mean'), source_to_donor_pair_flip_rate=('source_to_donor_pair_flip','mean')
).reindex(ORDER)
display(EVIDENCE_SUMMARY.round(4))
fig,axes=plt.subplots(1,3,figsize=(17,5))
for ax,column,scale,title in [
 (axes[0],'off_target_source_evidence',1,'A · Source-over-donor gap reduction after erasure'),
 (axes[1],'pairwise_source_share_reduction',100,'B · Pairwise source-share reduction (percentage points)'),
 (axes[2],'source_minus_donor_logit_before',1,'C · Source-over-donor species gap before erasure')]:
    vals=[EVIDENCE_ROWS.loc[EVIDENCE_ROWS.part==p,column].to_numpy()*scale for p in ORDER]
    boxes=ax.boxplot(vals,tick_labels=ORDER,patch_artist=True,showfliers=False,whis=(5,95))
    for patch,p in zip(boxes['boxes'],ORDER): patch.set_facecolor(COLORS[p]); patch.set_alpha(.65)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title,fontsize=9)
    ax.set_ylabel('percentage points' if scale==100 else 'species-logit units')
fig.suptitle(f'MCBM gamma={GAMMA:g} · Frozen-head off-target erasure, old/new slots untouched')
plt.tight_layout(); plt.show()

PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']=EVIDENCE_ROWS
PARITY_RUN.append(dict(figure='f8d-source',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f9-new

## 9 · Which measured information predicts new swaps, and what remains?

A five-fold diagnostic predicts two outcomes while every swap from one
original image stays in one held-out fold:

- final margin `m_cf`, scored by `RMSE = sqrt(mean((observed-predicted)^2))`;
- controlled event `1[response_delta>0 and m_cf<0]`, scored by Brier error
  `mean((observed 0/1 - predicted probability)^2)`.

Lower error is better. The full diagnostic receives part, starting
margin, visible pixels, source/donor support, source/donor label conflict,
ordinary exact-value recognition, and source species. One family is then
removed at a time while all others remain. A positive bar means held-out
error increased without that family, so it supplied predictive information
not replaced by the remaining fields. Zero or a negative value gives it
no unique credit in this diagnostic.

Example: full margin RMSE is 2.94. If removing visibility raises it to
3.14, the visibility bar is `3.14-2.94=0.20` margin units. This is not
“20% of backwash”: correlated inputs can substitute for one another, and
a one-seed predictor does not isolate causes.

### Figure 9 · Held-out predictive value of each measured contributor family


- **Method in one line:** A predeclared ridge margin model and logistic event model used five source-image-grouped folds; each bar removes one feature family from the full model, and the exact-value holdout table tests transfer to wholly unseen inserted values.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
show_standard('f9-new','0721a3c241465b4f636371425b2f02c1427100095d4ad8e9b0e12bf0850c88b9')


In [ ]:
# ALT: MCBM gamma=0: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
recognition=ordinary_value_recognition(z_saved,c_saved,SPANS)
descriptors=add_descriptors(S,CONFLICT,recognition)
PREDICTIVE,ROW_PREDICTIONS=prediction_audit(descriptors)
VALUE_HOLDOUT=value_holdout_audit(descriptors)
full=PREDICTIVE.loc[PREDICTIVE.model=="full measured set"].iloc[0]
omissions=PREDICTIVE[PREDICTIVE.model.str.startswith("full minus ")].copy()
if len(omissions)!=7: raise RuntimeError("expected seven one-family omission tests")
omissions["family"]=omissions.model.str.replace("full minus ","",regex=False)
omissions["RMSE_increase_when_omitted"]=omissions.margin_RMSE-full.margin_RMSE
omissions["Brier_increase_when_omitted"]=omissions.event_Brier-full.event_Brier
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
axes[0].barh(omissions.family,omissions.RMSE_increase_when_omitted,color="#4477AA")
axes[0].axvline(0,color="black",lw=.8)
axes[0].set_xlabel("increase in held-out RMSE when omitted")
axes[0].set_title(f"A · Final-margin prediction; full RMSE = {full.margin_RMSE:.3f}")
axes[1].barh(omissions.family,omissions.Brier_increase_when_omitted,color="#CC6677")
axes[1].axvline(0,color="black",lw=.8)
axes[1].set_xlabel("increase in held-out Brier error when omitted")
axes[1].set_title(f"B · Controlled-event prediction; full Brier = {full.event_Brier:.3f}")
fig.suptitle("Figure 9 · Predictive value of each measured contributor family")
plt.tight_layout(); plt.show()
display(PREDICTIVE.round(4))
print("Stress test: predict each inserted exact value after withholding every row that inserts it")
display(VALUE_HOLDOUT.round(4))

PARITY_CACHE[GAMMA]['PREDICTIVE']=PREDICTIVE
PARITY_CACHE[GAMMA]['VALUE_HOLDOUT']=VALUE_HOLDOUT
PARITY_RUN.append(dict(figure='f9-new',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
recognition=ordinary_value_recognition(z_saved,c_saved,SPANS)
descriptors=add_descriptors(S,CONFLICT,recognition)
PREDICTIVE,ROW_PREDICTIONS=prediction_audit(descriptors)
VALUE_HOLDOUT=value_holdout_audit(descriptors)
full=PREDICTIVE.loc[PREDICTIVE.model=="full measured set"].iloc[0]
omissions=PREDICTIVE[PREDICTIVE.model.str.startswith("full minus ")].copy()
if len(omissions)!=7: raise RuntimeError("expected seven one-family omission tests")
omissions["family"]=omissions.model.str.replace("full minus ","",regex=False)
omissions["RMSE_increase_when_omitted"]=omissions.margin_RMSE-full.margin_RMSE
omissions["Brier_increase_when_omitted"]=omissions.event_Brier-full.event_Brier
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
axes[0].barh(omissions.family,omissions.RMSE_increase_when_omitted,color="#4477AA")
axes[0].axvline(0,color="black",lw=.8)
axes[0].set_xlabel("increase in held-out RMSE when omitted")
axes[0].set_title(f"A · Final-margin prediction; full RMSE = {full.margin_RMSE:.3f}")
axes[1].barh(omissions.family,omissions.Brier_increase_when_omitted,color="#CC6677")
axes[1].axvline(0,color="black",lw=.8)
axes[1].set_xlabel("increase in held-out Brier error when omitted")
axes[1].set_title(f"B · Controlled-event prediction; full Brier = {full.event_Brier:.3f}")
fig.suptitle("Figure 9 · Predictive value of each measured contributor family")
plt.tight_layout(); plt.show()
display(PREDICTIVE.round(4))
print("Stress test: predict each inserted exact value after withholding every row that inserts it")
display(VALUE_HOLDOUT.round(4))

PARITY_CACHE[GAMMA]['PREDICTIVE']=PREDICTIVE
PARITY_CACHE[GAMMA]['VALUE_HOLDOUT']=VALUE_HOLDOUT
PARITY_RUN.append(dict(figure='f9-new',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
recognition=ordinary_value_recognition(z_saved,c_saved,SPANS)
descriptors=add_descriptors(S,CONFLICT,recognition)
PREDICTIVE,ROW_PREDICTIONS=prediction_audit(descriptors)
VALUE_HOLDOUT=value_holdout_audit(descriptors)
full=PREDICTIVE.loc[PREDICTIVE.model=="full measured set"].iloc[0]
omissions=PREDICTIVE[PREDICTIVE.model.str.startswith("full minus ")].copy()
if len(omissions)!=7: raise RuntimeError("expected seven one-family omission tests")
omissions["family"]=omissions.model.str.replace("full minus ","",regex=False)
omissions["RMSE_increase_when_omitted"]=omissions.margin_RMSE-full.margin_RMSE
omissions["Brier_increase_when_omitted"]=omissions.event_Brier-full.event_Brier
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
axes[0].barh(omissions.family,omissions.RMSE_increase_when_omitted,color="#4477AA")
axes[0].axvline(0,color="black",lw=.8)
axes[0].set_xlabel("increase in held-out RMSE when omitted")
axes[0].set_title(f"A · Final-margin prediction; full RMSE = {full.margin_RMSE:.3f}")
axes[1].barh(omissions.family,omissions.Brier_increase_when_omitted,color="#CC6677")
axes[1].axvline(0,color="black",lw=.8)
axes[1].set_xlabel("increase in held-out Brier error when omitted")
axes[1].set_title(f"B · Controlled-event prediction; full Brier = {full.event_Brier:.3f}")
fig.suptitle("Figure 9 · Predictive value of each measured contributor family")
plt.tight_layout(); plt.show()
display(PREDICTIVE.round(4))
print("Stress test: predict each inserted exact value after withholding every row that inserts it")
display(VALUE_HOLDOUT.round(4))

PARITY_CACHE[GAMMA]['PREDICTIVE']=PREDICTIVE
PARITY_CACHE[GAMMA]['VALUE_HOLDOUT']=VALUE_HOLDOUT
PARITY_RUN.append(dict(figure='f9-new',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
recognition=ordinary_value_recognition(z_saved,c_saved,SPANS)
descriptors=add_descriptors(S,CONFLICT,recognition)
PREDICTIVE,ROW_PREDICTIONS=prediction_audit(descriptors)
VALUE_HOLDOUT=value_holdout_audit(descriptors)
full=PREDICTIVE.loc[PREDICTIVE.model=="full measured set"].iloc[0]
omissions=PREDICTIVE[PREDICTIVE.model.str.startswith("full minus ")].copy()
if len(omissions)!=7: raise RuntimeError("expected seven one-family omission tests")
omissions["family"]=omissions.model.str.replace("full minus ","",regex=False)
omissions["RMSE_increase_when_omitted"]=omissions.margin_RMSE-full.margin_RMSE
omissions["Brier_increase_when_omitted"]=omissions.event_Brier-full.event_Brier
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
axes[0].barh(omissions.family,omissions.RMSE_increase_when_omitted,color="#4477AA")
axes[0].axvline(0,color="black",lw=.8)
axes[0].set_xlabel("increase in held-out RMSE when omitted")
axes[0].set_title(f"A · Final-margin prediction; full RMSE = {full.margin_RMSE:.3f}")
axes[1].barh(omissions.family,omissions.Brier_increase_when_omitted,color="#CC6677")
axes[1].axvline(0,color="black",lw=.8)
axes[1].set_xlabel("increase in held-out Brier error when omitted")
axes[1].set_title(f"B · Controlled-event prediction; full Brier = {full.event_Brier:.3f}")
fig.suptitle("Figure 9 · Predictive value of each measured contributor family")
plt.tight_layout(); plt.show()
display(PREDICTIVE.round(4))
print("Stress test: predict each inserted exact value after withholding every row that inserts it")
display(VALUE_HOLDOUT.round(4))

PARITY_CACHE[GAMMA]['PREDICTIVE']=PREDICTIVE
PARITY_CACHE[GAMMA]['VALUE_HOLDOUT']=VALUE_HOLDOUT
PARITY_RUN.append(dict(figure='f9-new',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
recognition=ordinary_value_recognition(z_saved,c_saved,SPANS)
descriptors=add_descriptors(S,CONFLICT,recognition)
PREDICTIVE,ROW_PREDICTIONS=prediction_audit(descriptors)
VALUE_HOLDOUT=value_holdout_audit(descriptors)
full=PREDICTIVE.loc[PREDICTIVE.model=="full measured set"].iloc[0]
omissions=PREDICTIVE[PREDICTIVE.model.str.startswith("full minus ")].copy()
if len(omissions)!=7: raise RuntimeError("expected seven one-family omission tests")
omissions["family"]=omissions.model.str.replace("full minus ","",regex=False)
omissions["RMSE_increase_when_omitted"]=omissions.margin_RMSE-full.margin_RMSE
omissions["Brier_increase_when_omitted"]=omissions.event_Brier-full.event_Brier
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
axes[0].barh(omissions.family,omissions.RMSE_increase_when_omitted,color="#4477AA")
axes[0].axvline(0,color="black",lw=.8)
axes[0].set_xlabel("increase in held-out RMSE when omitted")
axes[0].set_title(f"A · Final-margin prediction; full RMSE = {full.margin_RMSE:.3f}")
axes[1].barh(omissions.family,omissions.Brier_increase_when_omitted,color="#CC6677")
axes[1].axvline(0,color="black",lw=.8)
axes[1].set_xlabel("increase in held-out Brier error when omitted")
axes[1].set_title(f"B · Controlled-event prediction; full Brier = {full.event_Brier:.3f}")
fig.suptitle("Figure 9 · Predictive value of each measured contributor family")
plt.tight_layout(); plt.show()
display(PREDICTIVE.round(4))
print("Stress test: predict each inserted exact value after withholding every row that inserts it")
display(VALUE_HOLDOUT.round(4))

PARITY_CACHE[GAMMA]['PREDICTIVE']=PREDICTIVE
PARITY_CACHE[GAMMA]['VALUE_HOLDOUT']=VALUE_HOLDOUT
PARITY_RUN.append(dict(figure='f9-new',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Two held-out omission panels showing the increase in final-margin and controlled-event prediction error when each measured contributor family is removed.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
recognition=ordinary_value_recognition(z_saved,c_saved,SPANS)
descriptors=add_descriptors(S,CONFLICT,recognition)
PREDICTIVE,ROW_PREDICTIONS=prediction_audit(descriptors)
VALUE_HOLDOUT=value_holdout_audit(descriptors)
full=PREDICTIVE.loc[PREDICTIVE.model=="full measured set"].iloc[0]
omissions=PREDICTIVE[PREDICTIVE.model.str.startswith("full minus ")].copy()
if len(omissions)!=7: raise RuntimeError("expected seven one-family omission tests")
omissions["family"]=omissions.model.str.replace("full minus ","",regex=False)
omissions["RMSE_increase_when_omitted"]=omissions.margin_RMSE-full.margin_RMSE
omissions["Brier_increase_when_omitted"]=omissions.event_Brier-full.event_Brier
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
axes[0].barh(omissions.family,omissions.RMSE_increase_when_omitted,color="#4477AA")
axes[0].axvline(0,color="black",lw=.8)
axes[0].set_xlabel("increase in held-out RMSE when omitted")
axes[0].set_title(f"A · Final-margin prediction; full RMSE = {full.margin_RMSE:.3f}")
axes[1].barh(omissions.family,omissions.Brier_increase_when_omitted,color="#CC6677")
axes[1].axvline(0,color="black",lw=.8)
axes[1].set_xlabel("increase in held-out Brier error when omitted")
axes[1].set_title(f"B · Controlled-event prediction; full Brier = {full.event_Brier:.3f}")
fig.suptitle("Figure 9 · Predictive value of each measured contributor family")
plt.tight_layout(); plt.show()
display(PREDICTIVE.round(4))
print("Stress test: predict each inserted exact value after withholding every row that inserts it")
display(VALUE_HOLDOUT.round(4))

PARITY_CACHE[GAMMA]['PREDICTIVE']=PREDICTIVE
PARITY_CACHE[GAMMA]['VALUE_HOLDOUT']=VALUE_HOLDOUT
PARITY_RUN.append(dict(figure='f9-new',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f9b

## 9b · Synthesis only: do the already measured contributors line up with the controlled part ordering?

**Question.** Synthesis only: do the already measured contributors line up with the controlled part ordering?

**Variables and prediction.** Place four separately defined part-level quantities in aligned panels: the controlled backwash-candidate rate, the same rate among swaps with at least 100 target pixels, the training label/mask conflict rate, and one minus exact donor-value recognition. Tail should be high across several contributor panels while wing and foot should be low if the proposed explanation matches the controlled outcome. The panels use different units and must not be added together.

**Method.** Use the same five-part order in every panel and print the exact table.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 9b · Synthesis only: do the already measured contributors line up with the controlled part ordering?

**How to read the figure.** All four panels use the same y-axis part order. Panel A is the fraction of
all swaps satisfying `response_delta>0 and m_cf<0`. Panel B repeats that
fraction only when the inserted target occupies at least 100 pixels. Panel C
is the fraction of original positive training labels removed by the matched
visibility rule. Panel D is one minus the post-swap inserted-value recognition
rate. Larger is worse in every panel, but the denominators and meanings differ,
so the bar heights must not be added. The shared ordering asks whether the
proposed contributors align with the controlled outcome.


- **Method in one line:** We aligned four previously computed part-level summaries in one fixed order without fitting or adding them, so only their descriptive ordering—not percent explained—can be compared.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
show_standard('f9b','68ccde0ac74548185b9b035b646160413d93d1cd175dc1291f71cb3d439c600d')


In [ ]:
# ALT: MCBM gamma=0: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · MODEL RECOGNITION: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))

PARITY_RUN.append(dict(figure='f9b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · MODEL RECOGNITION: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))

PARITY_RUN.append(dict(figure='f9b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · MODEL RECOGNITION: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))

PARITY_RUN.append(dict(figure='f9b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · MODEL RECOGNITION: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))

PARITY_RUN.append(dict(figure='f9b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · MODEL RECOGNITION: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))

PARITY_RUN.append(dict(figure='f9b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
diag=PARITY_CACHE[GAMMA]['diag']
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · MODEL RECOGNITION: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))

PARITY_RUN.append(dict(figure='f9b',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction f10

## 10 · Is final concept margin associated with donor-species probability?

**Question.** Is final concept margin associated with donor-species probability?

**Variables and prediction.** Relate final concept margin to the model's donor-species probability, which is a different downstream quantity; this is an association, not an intervention on the margin. A small downstream change would limit the harm to explanation reliability rather than widespread class failure.

**Method.** Use independent final-margin bins and print bin counts.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 10 · Is final concept margin associated with donor-species probability?

**How to read the figure.** Swaps are divided into ten non-overlapping, approximately equal-count bins by final donor-minus-source concept
margin on the x-axis. The y-axis is the model's mean probability for the donor
species, with the number of rows printed per bin. This asks whether concept
grounding failure is associated with a downstream class quantity; it is intentionally the one
place where class probability, rather than raw concept `z`, is the outcome.


**Concrete example.** Suppose a red tail is replaced by a blue tail.
“Donor-positive concept margin” means the blue-tail raw score finishes
above the red-tail raw score. The **donor species** is the complete species
that supplied the blue tail. Its saved probability is the CBM class head's
probability for that whole donor bird—not a probability that the tail is
blue. Because the body and the other four parts still belong to the source
bird, a correct blue-tail win need not make the complete donor species
likely. This figure asks only whether the two quantities move together.


- **Method in one line:** We divided all swaps into ten disjoint equal-count bins by final concept margin and averaged the frozen CBM's saved donor-species probability within each bin; no regression or new classifier was fitted.


**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
show_standard('f10','796bb0af33395c3f929f0835af886d661bc7a0727ec83e673f4216ed53faf4d6')


In [ ]:
# ALT: MCBM gamma=0: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))

PARITY_RUN.append(dict(figure='f10',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))

PARITY_RUN.append(dict(figure='f10',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))

PARITY_RUN.append(dict(figure='f10',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))

PARITY_RUN.append(dict(figure='f10',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))

PARITY_RUN.append(dict(figure='f10',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))

PARITY_RUN.append(dict(figure='f10',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Standard construction app-evidence-correlation-code



**Inputs:** frozen Standard baseline, then official MCBM seed 1 at each declared gamma. The adapters below change the model/data arrays, not the pixel intervention. Any raw-score axis remains in that model’s own units. Read the printed denominators; a correlation is not a causal attribution.


In [ ]:
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
show_standard('app-evidence-correlation-code','4ec482678a79180e8bcdb3ff7ed5f1bf867b85a5c5edd2a8116053edd75c33d9')


In [ ]:
# ALT: MCBM gamma=0: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
GAMMA=0.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
EVIDENCE_ROWS=PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
PAIR_KEYS=["part","source_value","donor_value"]
EVIDENCE_ROWS["e_within_pair"]=(
    EVIDENCE_ROWS.off_target_source_evidence-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).off_target_source_evidence.transform("mean"))
EVIDENCE_ROWS["margin_within_pair"]=(
    EVIDENCE_ROWS.m_cf-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).m_cf.transform("mean"))
correlation_rows=[]; fifth_rows=[]
for part in ORDER:
    part_rows=EVIDENCE_ROWS[EVIDENCE_ROWS.part==part].copy()
    rho=float(part_rows.e_within_pair.rank().corr(
        part_rows.margin_within_pair.rank()))
    correlation_rows.append({"part":part,"n_swaps":len(part_rows),
                             "n_original_images":part_rows.original_image.nunique(),
                             "within_pair_spearman_rho":rho})
    part_rows["evidence_fifth"]=pd.qcut(
        part_rows.e_within_pair.rank(method="first"),5,labels=False)+1
    for fifth,group in part_rows.groupby("evidence_fifth"):
        fifth_rows.append({"part":part,"evidence_fifth":int(fifth),
            "n_swaps":len(group),
            "n_original_images":group.original_image.nunique(),
            "mean_e_within_pair":group.e_within_pair.mean(),
            "mean_margin_within_pair":group.margin_within_pair.mean(),
            "controlled_event_rate":group.controlled_event.mean()})
EVIDENCE_CORRELATION=pd.DataFrame(correlation_rows)
EVIDENCE_FIFTHS=pd.DataFrame(fifth_rows)
display(EVIDENCE_CORRELATION.round(4)); display(EVIDENCE_FIFTHS.round(4))

PARITY_RUN.append(dict(figure='app-evidence-correlation-code',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.1: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
GAMMA=0.1
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
EVIDENCE_ROWS=PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
PAIR_KEYS=["part","source_value","donor_value"]
EVIDENCE_ROWS["e_within_pair"]=(
    EVIDENCE_ROWS.off_target_source_evidence-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).off_target_source_evidence.transform("mean"))
EVIDENCE_ROWS["margin_within_pair"]=(
    EVIDENCE_ROWS.m_cf-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).m_cf.transform("mean"))
correlation_rows=[]; fifth_rows=[]
for part in ORDER:
    part_rows=EVIDENCE_ROWS[EVIDENCE_ROWS.part==part].copy()
    rho=float(part_rows.e_within_pair.rank().corr(
        part_rows.margin_within_pair.rank()))
    correlation_rows.append({"part":part,"n_swaps":len(part_rows),
                             "n_original_images":part_rows.original_image.nunique(),
                             "within_pair_spearman_rho":rho})
    part_rows["evidence_fifth"]=pd.qcut(
        part_rows.e_within_pair.rank(method="first"),5,labels=False)+1
    for fifth,group in part_rows.groupby("evidence_fifth"):
        fifth_rows.append({"part":part,"evidence_fifth":int(fifth),
            "n_swaps":len(group),
            "n_original_images":group.original_image.nunique(),
            "mean_e_within_pair":group.e_within_pair.mean(),
            "mean_margin_within_pair":group.margin_within_pair.mean(),
            "controlled_event_rate":group.controlled_event.mean()})
EVIDENCE_CORRELATION=pd.DataFrame(correlation_rows)
EVIDENCE_FIFTHS=pd.DataFrame(fifth_rows)
display(EVIDENCE_CORRELATION.round(4)); display(EVIDENCE_FIFTHS.round(4))

PARITY_RUN.append(dict(figure='app-evidence-correlation-code',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=0.3: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
GAMMA=0.3
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
EVIDENCE_ROWS=PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
PAIR_KEYS=["part","source_value","donor_value"]
EVIDENCE_ROWS["e_within_pair"]=(
    EVIDENCE_ROWS.off_target_source_evidence-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).off_target_source_evidence.transform("mean"))
EVIDENCE_ROWS["margin_within_pair"]=(
    EVIDENCE_ROWS.m_cf-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).m_cf.transform("mean"))
correlation_rows=[]; fifth_rows=[]
for part in ORDER:
    part_rows=EVIDENCE_ROWS[EVIDENCE_ROWS.part==part].copy()
    rho=float(part_rows.e_within_pair.rank().corr(
        part_rows.margin_within_pair.rank()))
    correlation_rows.append({"part":part,"n_swaps":len(part_rows),
                             "n_original_images":part_rows.original_image.nunique(),
                             "within_pair_spearman_rho":rho})
    part_rows["evidence_fifth"]=pd.qcut(
        part_rows.e_within_pair.rank(method="first"),5,labels=False)+1
    for fifth,group in part_rows.groupby("evidence_fifth"):
        fifth_rows.append({"part":part,"evidence_fifth":int(fifth),
            "n_swaps":len(group),
            "n_original_images":group.original_image.nunique(),
            "mean_e_within_pair":group.e_within_pair.mean(),
            "mean_margin_within_pair":group.margin_within_pair.mean(),
            "controlled_event_rate":group.controlled_event.mean()})
EVIDENCE_CORRELATION=pd.DataFrame(correlation_rows)
EVIDENCE_FIFTHS=pd.DataFrame(fifth_rows)
display(EVIDENCE_CORRELATION.round(4)); display(EVIDENCE_FIFTHS.round(4))

PARITY_RUN.append(dict(figure='app-evidence-correlation-code',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=1: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
GAMMA=1.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
EVIDENCE_ROWS=PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
PAIR_KEYS=["part","source_value","donor_value"]
EVIDENCE_ROWS["e_within_pair"]=(
    EVIDENCE_ROWS.off_target_source_evidence-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).off_target_source_evidence.transform("mean"))
EVIDENCE_ROWS["margin_within_pair"]=(
    EVIDENCE_ROWS.m_cf-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).m_cf.transform("mean"))
correlation_rows=[]; fifth_rows=[]
for part in ORDER:
    part_rows=EVIDENCE_ROWS[EVIDENCE_ROWS.part==part].copy()
    rho=float(part_rows.e_within_pair.rank().corr(
        part_rows.margin_within_pair.rank()))
    correlation_rows.append({"part":part,"n_swaps":len(part_rows),
                             "n_original_images":part_rows.original_image.nunique(),
                             "within_pair_spearman_rho":rho})
    part_rows["evidence_fifth"]=pd.qcut(
        part_rows.e_within_pair.rank(method="first"),5,labels=False)+1
    for fifth,group in part_rows.groupby("evidence_fifth"):
        fifth_rows.append({"part":part,"evidence_fifth":int(fifth),
            "n_swaps":len(group),
            "n_original_images":group.original_image.nunique(),
            "mean_e_within_pair":group.e_within_pair.mean(),
            "mean_margin_within_pair":group.margin_within_pair.mean(),
            "controlled_event_rate":group.controlled_event.mean()})
EVIDENCE_CORRELATION=pd.DataFrame(correlation_rows)
EVIDENCE_FIFTHS=pd.DataFrame(fifth_rows)
display(EVIDENCE_CORRELATION.round(4)); display(EVIDENCE_FIFTHS.round(4))

PARITY_RUN.append(dict(figure='app-evidence-correlation-code',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=3: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
GAMMA=3.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
EVIDENCE_ROWS=PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
PAIR_KEYS=["part","source_value","donor_value"]
EVIDENCE_ROWS["e_within_pair"]=(
    EVIDENCE_ROWS.off_target_source_evidence-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).off_target_source_evidence.transform("mean"))
EVIDENCE_ROWS["margin_within_pair"]=(
    EVIDENCE_ROWS.m_cf-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).m_cf.transform("mean"))
correlation_rows=[]; fifth_rows=[]
for part in ORDER:
    part_rows=EVIDENCE_ROWS[EVIDENCE_ROWS.part==part].copy()
    rho=float(part_rows.e_within_pair.rank().corr(
        part_rows.margin_within_pair.rank()))
    correlation_rows.append({"part":part,"n_swaps":len(part_rows),
                             "n_original_images":part_rows.original_image.nunique(),
                             "within_pair_spearman_rho":rho})
    part_rows["evidence_fifth"]=pd.qcut(
        part_rows.e_within_pair.rank(method="first"),5,labels=False)+1
    for fifth,group in part_rows.groupby("evidence_fifth"):
        fifth_rows.append({"part":part,"evidence_fifth":int(fifth),
            "n_swaps":len(group),
            "n_original_images":group.original_image.nunique(),
            "mean_e_within_pair":group.e_within_pair.mean(),
            "mean_margin_within_pair":group.margin_within_pair.mean(),
            "controlled_event_rate":group.controlled_event.mean()})
EVIDENCE_CORRELATION=pd.DataFrame(correlation_rows)
EVIDENCE_FIFTHS=pd.DataFrame(fifth_rows)
display(EVIDENCE_CORRELATION.round(4)); display(EVIDENCE_FIFTHS.round(4))

PARITY_RUN.append(dict(figure='app-evidence-correlation-code',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


In [ ]:
# ALT: MCBM gamma=5: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
GAMMA=5.0
ordinary=HEALTH_DATA[(GAMMA,1)]
h_saved,z_saved,c_saved,y_saved=ordinary['h'],ordinary['z'],ordinary['c'],ordinary['y']
y_pred_saved=ordinary['y_probability'].argmax(1)
S=SW[(SW.gamma==GAMMA)&(SW.seed==1)].copy().reset_index(drop=True)
S['donor_gain']=S.z_new-S.z_new_orig
S['source_decrease']=S.z_old_orig-S.z_old
S['responded_but_source_wins']=(S.response_delta>0)&(S.m_cf<0)
S['controlled_event']=S.responded_but_source_wins.astype(int)
checkpoint=REPO/'external/minimal_cbm/results'/f'funnybirds-mcbm-g{checkpoint_tag(GAMMA)}'/'1/models/epoch_100.pt'
saved_head=task_head(checkpoint)
print('MCBM gamma=',GAMMA,'ordinary images=',len(y_saved),'swaps=',len(S),'originals=',S.orig_render_id.nunique(),flush=True)
EVIDENCE_ROWS=PARITY_CACHE[GAMMA]['EVIDENCE_ROWS']
# ALT: Two audit tables retaining the earlier within-exact-pair rank-correlation and evidence-fifths analysis as secondary exploratory evidence; no model is fitted and no causal claim is made.
PAIR_KEYS=["part","source_value","donor_value"]
EVIDENCE_ROWS["e_within_pair"]=(
    EVIDENCE_ROWS.off_target_source_evidence-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).off_target_source_evidence.transform("mean"))
EVIDENCE_ROWS["margin_within_pair"]=(
    EVIDENCE_ROWS.m_cf-
    EVIDENCE_ROWS.groupby(PAIR_KEYS).m_cf.transform("mean"))
correlation_rows=[]; fifth_rows=[]
for part in ORDER:
    part_rows=EVIDENCE_ROWS[EVIDENCE_ROWS.part==part].copy()
    rho=float(part_rows.e_within_pair.rank().corr(
        part_rows.margin_within_pair.rank()))
    correlation_rows.append({"part":part,"n_swaps":len(part_rows),
                             "n_original_images":part_rows.original_image.nunique(),
                             "within_pair_spearman_rho":rho})
    part_rows["evidence_fifth"]=pd.qcut(
        part_rows.e_within_pair.rank(method="first"),5,labels=False)+1
    for fifth,group in part_rows.groupby("evidence_fifth"):
        fifth_rows.append({"part":part,"evidence_fifth":int(fifth),
            "n_swaps":len(group),
            "n_original_images":group.original_image.nunique(),
            "mean_e_within_pair":group.e_within_pair.mean(),
            "mean_margin_within_pair":group.margin_within_pair.mean(),
            "controlled_event_rate":group.controlled_event.mean()})
EVIDENCE_CORRELATION=pd.DataFrame(correlation_rows)
EVIDENCE_FIFTHS=pd.DataFrame(fifth_rows)
display(EVIDENCE_CORRELATION.round(4)); display(EVIDENCE_FIFTHS.round(4))

PARITY_RUN.append(dict(figure='app-evidence-correlation-code',model=f'MCBM gamma={GAMMA:g}',status='analysis executed; interpretation pending visual review'))


**Reading rule after this figure.** The displayed tables and labels are the current literal results; no Standard conclusion is assumed. Compare every part against gamma zero. Record improvement, worsening, ties, and unavailable strata separately. A model difference supports only this measured association/intervention; altered score scale, optimization, and a single causal seed remain alternatives. Independent frozen-render seed replay distinguishes reproducibility. The next figure tests the next step, not an explanation already established here.


## Supporting appendix: species-matched recognition

This is a health/species-dependence diagnostic, not a substitute for the controlled swaps.


In [ ]:
# ALT: Ordinary MCBM populations for the matched-recall appendix.
MODEL_DATA={f"g={g:g}":HEALTH_DATA[(g,1)] for g in GAMMAS}


## 11 · Is recognition of the same positive concept species-dependent?

**Notebook 02 connection.** This restores the authoritative FunnyBird recall
diagnostic as supporting evidence. It does not replace notebook 02's controlled swap.

**Question.** For the same exact positive concept, does recognition differ
between species after positive and negative sample counts are matched?

**Variables and prediction.** The authoritative `fb_recallv2` method has two stages. First, for an exact
concept, it pairs two species only when each contains at least ten positive and
ten negative rows; positive and negative sample counts are matched between the
species. Only if this produces no pairs does it use the all-positive-species
fallback. This notebook prints the selected rule and eligibility coverage.

The current curated validation labels vary within species, so the expected rule
is `matched_positive_negative`, not the fallback. In 300 vectorized bootstrap
runs per pair, `recall gap` is the absolute difference in `P(z>0 | c=1)`.
`balanced-accuracy gap` also uses the matched negatives. `raw-z gap` is the
absolute difference in mean positive `z`, standardized within model/concept.
Zero means equal recognition.

Each heatmap cell is the median across valid concept/species pairs assigned to
that part. Images and pairs are not independent model seeds. Recall is a model-
health/species-dependence diagnostic; the controlled replacement remains the
grounding test.

**Method.** Use 300 vectorized bootstrap draws per eligible species pair, print
the chosen pairing rule and coverage, and keep all models on the same prediction
population. Bootstrap pairs are not independent trained seeds.

### Figure 11 · Matched-species recall, balanced accuracy, and raw-logit gaps

**How to read the figure.** Rows are the six MCBM gammas;
columns are parts. Panel A is the absolute positive-recall difference, Panel B
the absolute balanced-accuracy difference, and Panel C the positive raw-logit
difference measured in within-concept standard deviations. Zero means equal
recognition across the paired species. Example: raw-`z` gap `0.15` means the two
species' mean positive scores differ by 0.15 within-concept standard deviations,
even if both stay above the `z=0` threshold and recall barely changes.


In [ ]:
# ALT: Figure 11. Two-stage matched-species recall, balanced-accuracy, and standardized raw-logit gaps for every MCBM gamma.
from itertools import combinations
rec=[]; coverage=[]; B_RECALL=300
for model,d in MODEL_DATA.items():
 gamma=float(model.split("=")[1]); y=d["y"]; c=d["c"].astype(int); z=d["z"]
 for j,name in enumerate(CONCEPT_NAMES):
  zj=z[:,j]; zstd=(zj-zj.mean())/(zj.std()+1e-12); stats=[]
  for sp in np.unique(y):
   ix=y==sp; n=int(ix.sum()); npos=int(c[ix,j].sum()); stats.append((int(sp),n,npos,n-npos,npos/n))
  eligible=[s for s,n,np_,nn,p in stats if np_>=10 and nn>=10]
  rule="matched_positive_negative"
  pairs=list(combinations(eligible,2))[:200]
  if not pairs:
   eligible=[s for s,n,np_,nn,p in stats if np_>=3 and p>=.9]
   rule="all_positive_fallback"; pairs=list(combinations(eligible,2))[:200]
  coverage.append(dict(model=model,concept=name,part=CONCEPT_PART[name],pairing_rule=rule,
                       eligible_species=len(eligible),pairs=len(pairs),max_species_prevalence=max(s[-1] for s in stats)))
  for pair_index,(a,b) in enumerate(pairs):
   Apos=np.where((y==a)&(c[:,j]==1))[0]; Bpos=np.where((y==b)&(c[:,j]==1))[0]
   Aneg=np.where((y==a)&(c[:,j]==0))[0]; Bneg=np.where((y==b)&(c[:,j]==0))[0]
   mpos=min(len(Apos),len(Bpos)); mneg=min(len(Aneg),len(Bneg))
   rng=np.random.default_rng(20260806+j*1000+pair_index)
   ap=Apos[rng.integers(len(Apos),size=(B_RECALL,mpos))]; bp=Bpos[rng.integers(len(Bpos),size=(B_RECALL,mpos))]
   recA=(zj[ap]>0).mean(1); recB=(zj[bp]>0).mean(1); recall_gaps=np.abs(recA-recB)
   raw_gaps=np.abs(zstd[ap].mean(1)-zstd[bp].mean(1))
   if rule=="matched_positive_negative":
    an=Aneg[rng.integers(len(Aneg),size=(B_RECALL,mneg))]; bn=Bneg[rng.integers(len(Bneg),size=(B_RECALL,mneg))]
    baA=.5*(recA+(zj[an]<=0).mean(1)); baB=.5*(recB+(zj[bn]<=0).mean(1)); ba_gap=float(np.abs(baA-baB).mean())
   else: ba_gap=np.nan
   rec.append(dict(model=model,gamma=gamma,seed=1,concept=name,part=CONCEPT_PART[name],species_a=a,species_b=b,
      pairing_rule=rule,n_positive=mpos,n_negative=mneg,recall_gap=float(recall_gaps.mean()),
      recall_gap_ci_low=float(np.quantile(recall_gaps,.025)),recall_gap_ci_high=float(np.quantile(recall_gaps,.975)),
      balanced_accuracy_gap=ba_gap,standardized_raw_z_gap=float(raw_gaps.mean())))
RECALL=pd.DataFrame(rec)
COVERAGE=pd.DataFrame(coverage)
display(COVERAGE.groupby(["model","pairing_rule"]).agg(concepts=("concept","nunique"),eligible_species_median=("eligible_species","median"),pairs=("pairs","sum"),maximum_prevalence=("max_species_prevalence","max")).round(3))
if RECALL.empty: raise RuntimeError("authoritative two-stage recall pairing produced no pairs; inspect displayed coverage")
model_order=[f"g={g:g}" for g in GAMMAS]
Rg=RECALL.groupby(["model","part"])[["recall_gap","balanced_accuracy_gap","standardized_raw_z_gap"]].median()
R1=Rg.recall_gap.unstack().reindex(index=model_order,columns=ORDER)
RB=Rg.balanced_accuracy_gap.unstack().reindex(index=model_order,columns=ORDER)
R2=Rg.standardized_raw_z_gap.unstack().reindex(index=model_order,columns=ORDER)
fig,ax=plt.subplots(1,3,figsize=(18,4))
heat(ax[0],R1,"Median matched-species positive-recall gap","absolute recall difference",0,1,"magma")
heat(ax[1],RB,"Median matched-species balanced-accuracy gap","absolute BA difference",0,1,"magma")
heat(ax[2],R2,"Median matched-species standardized raw-z gap","within-concept SD units",0,None,"viridis")
plt.tight_layout(); display(RECALL.groupby(["model","part","pairing_rule"]).agg(pairs=("recall_gap","size"),median_recall_gap=("recall_gap","median"),median_balanced_accuracy_gap=("balanced_accuracy_gap","median"),median_raw_z_gap=("standardized_raw_z_gap","median")).round(3))


### Review record for Figure 11

**INCOMPLETE: current output requires execution and visual review.**

- **Literal result:** Record the actual values, sample sizes and exceptions.
- **What it supports:** State only the claim measured by this figure.
- **Plausible alternative:** Give a concrete competing explanation.
- **Discriminating test:** State which observation would distinguish it.
- **Next question:** Explain why the next analysis follows from this result.

Display the complete current figure in chat before filling this record.


## Loss-engineering diagnostic · Which objectives currently push h in competing directions?

**Question/prediction:** if task and concept supervision compete at the internal
slot, their gradients can point in opposite directions. Compression may dominate
or be small at the final checkpoint. Neither pattern is assumed beforehand.

**Inputs/model:** each gamma's frozen official MCBM, saved ordinary h,c,y, its
actual concept-loss positive weights and β/γ. No diagnostic classifier is trained.
Autograd differentiates only copies of h; trained parameters are frozen and no
optimizer or parameter update exists. We use the implementation's training-noise
rule, one deterministic Normal draw per image (seed20260903), not an observed
historical training batch. Batch64 means are multiplied by their actual batch size
to express per-image derivatives, including the short final batch.

Let a=∂L_task/∂h, b=∂(βL_concept)/∂h, r=∂(γL_rep)/∂h. For every part, A/B/C show
sqrt(mean_(image,coordinate) gradient²): gradient RMS per coordinate, not summed
over a wider part. D shows mean cosine(a_part,b_part)=a·b/(||a||||b||) on images
where both norms exceed the declared denominator tolerance1e−12; counts and the
fraction with cosine<0 are printed. Undefined zero-gradient cosines remain missing.
Rows are gamma; columns are the same five parts. Larger RMS means a stronger
local push, not a larger historical causal contribution. D ranges−1 to+1.

Example: a=[1,0],b=[−2,0] gives cosine−1: descending one objective locally raises
the other to first order. a=[1,0],b=[0,1] gives0: orthogonal pushes. A tiny norm
can mean a satisfied objective, saturation or a flat reader, so consult health
and q_j slopes before calling it successful compression or broken learning.


In [ ]:
# ALT: Frozen-checkpoint official loss gradients by part and gamma; not a training trajectory.
from mcbm_loss_report import loss_gradient_audit
LOSS_GRADIENTS=pd.concat([loss_gradient_audit(
    HEALTH_DATA[(g,1)]['h'],HEALTH_DATA[(g,1)]['c'],HEALTH_DATA[(g,1)]['y'],g,SPANS
) for g in GAMMAS],ignore_index=True)
fig,axes=plt.subplots(1,4,figsize=(19,4.8))
for ax,column,title in zip(axes,
 ['task_gradient_RMS','concept_gradient_RMS','weighted_compression_gradient_RMS','task_concept_cosine_mean'],
 ['A · Task gradient','B · Weighted concept gradient','C · Weighted compression gradient','D · Task/concept alignment']):
    values=LOSS_GRADIENTS.pivot(index='gamma',columns='part',values=column).reindex(index=GAMMAS,columns=ORDER)
    heat(ax,values,title,'cosine' if column.endswith('mean') else 'gradient RMS per coordinate',
         -1 if column.endswith('mean') else 0,1 if column.endswith('mean') else None,
         'coolwarm' if column.endswith('mean') else 'viridis',fmt='.3f')
plt.tight_layout();plt.show();display(LOSS_GRADIENTS.round(5))


**What this can support:** the displayed gradient magnitudes and directions describe incentives at these frozen checkpoints. **Alternative:** another noise draw or earlier training epoch may differ. **Discriminator:** repeat noise draws and inspect saved training checkpoints under a matched recipe. **Next:** select a candidate loss only when the physical-swap and health results identify the failure it should address.


## What would justify a better loss—not merely another graph?

The original objective is a penalty on **what h resembles**, not on **where its
evidence came from**. Its coordinate gradient is 0.4γ(h_j−(6c_j−3)) per image
before batch averaging. A label-positive hidden tail and a visible tail receive
the same target. Making h perfectly±3 therefore does not guarantee named pixels
were used. Conversely, suppressing all variation can discard useful visibility
information. These are possible incentives, not established causes of our results.

Koh's concept BCE gradient (unweighted example) is sigmoid(z)−c: for c=1,
z=0 gives−.5 but z=5 gives about−.0067. It rewards correct confident labels without
choosing one common positive magnitude. MCBM's squared penalty adds that pressure
on h, but a learned q_j can amplify or flatten small changes. A nonlinear species
reader can also use interactions. Thus inspect h compression, q_j sensitivity,
z recognition, physical response, and the original reader separately.

| Measured failure, if present | Candidate change to test next | Exact additional term / needed data | What could still go wrong? |
|---|---|---|---|
| h compresses but inserted concepts still lose | add correct physical-swap ranking | λ mean max(0,κ−[z_d,cf−z_s,cf]); known valid donor/source swaps, κ>0 declared before training | ranking may overfit rendered edits; check untouched parts and ordinary health |
| score changes on a part whose pixels were untouched | enforce off-target invariance | λ mean sum_(j outside replaced part)(z_j,cf−z_j,orig)^2 | legitimate interactions/occlusion may change those pixels; validate masks first |
| positive hidden labels associate with poor response | visibility-aware targets (RLv2 chapter) | replace the declared hidden-positive targets, train under the matched protocol | other contextual shortcuts survive; do not declare all residuals explained |
| variation in h vanishes but z becomes flat/miscalibrated | test a simpler or calibrated concept reader | compare fixed monotone q_j with learned q_j under a matched recipe | changes architecture too; not a loss-only causal comparison |
| ordinary z magnitudes vary within the same answer | direct z prototype penalty as a separate ablation | λ mean (z_j−a(2c_j−1))², e.g. a=5 | a neat ±5 answer can still come entirely from species context |

These are **proposed training experiments**, not implemented or validated repairs.
This notebook trains diagnostics only and reuses frozen CBMs/MCBMs. No new scientific
model is submitted. A useful conclusion can be “compression changed X but did not
fix Y; measured failure Y motivates test Z,” without pretending every cause is known.

**Paper boundary:** the MCBM conditional-information motivation and its decoding,
disentanglement and internal-correction tests concern representation properties.
They do not by themselves establish physical part grounding. A weak diagnostic
does not prove every decoder fails; lower task accuracy can reflect optimization.
Gamma0 remains the closest MCBM baseline, but independent initialization and training
outcomes prevent a single-seed sweep from being a pure, replicated causal loss estimate.

**Next chapters:** RLv2 tests the label-conflict intervention, then CUB70 tests how
far these questions survive with released masks but no accepted native donor swap.
Neither observational mask associations nor uncalibrated pasted images inherit
FunnyBird's renderer-quality causal claim.


In [ ]:
# ALT: Loss-engineering decision table from current gamma outputs and an explicit execution parity checklist.
decision_rows=[]
for g in GAMMAS:
    s=SW[(SW.gamma==g)&(SW.seed==1)]
    ordinary_health=H[(H.gamma==g)&(H.seed==1)].iloc[0]
    for part in ORDER:
        q=s[s.part==part]
        decision_rows.append(dict(gamma=g,part=part,n=len(q),
            internal_target_RMSE=ordinary_health.target_rmse,
            ordinary_species_accuracy=ordinary_health.species_accuracy,
            donor_finishes_higher=float((q.m_cf>0).mean()),
            donorward_but_source_wins=float(((q.response_delta>0)&(q.m_cf<0)).mean()),
            no_donorward_move=float((q.response_delta<=0).mean()),
            final_ties=float((q.m_cf==0).mean()),
            inserted_value_accuracy=PARITY_CACHE[g]['diag'][part]))
LOSS_DECISIONS=pd.DataFrame(decision_rows)
display(LOSS_DECISIONS.round(4))
display(pd.DataFrame(PARITY_RUN))
print('Computation finished. Scientific interpretation remains pending review of EVERY current image; this is not a trained-model replication claim.')
report_dir=CURATED/'mcbm_notebook03_tables'
report_dir.mkdir(parents=True,exist_ok=True)
LOSS_DECISIONS.to_csv(report_dir/'loss_decisions.csv',index=False)
LOSS_GRADIENTS.to_csv(report_dir/'loss_gradients.csv',index=False)
pd.DataFrame(PARITY_RUN).to_csv(report_dir/'figure_execution_checklist.csv',index=False)
for gamma,results in PARITY_CACHE.items():
    for name,value in results.items():
        if isinstance(value,pd.DataFrame):
            value.to_csv(report_dir/f'g{checkpoint_tag(gamma)}_{name}.csv',index=False)
print('Current source tables:',report_dir)
